# Насколько точно мы знаем среднее?
## Занятие 8 — доверительный интервал и бутстрап

По 100 доставкам получили среднее время. Другая сотня заказов дала бы немного другой ответ.
**Как показать точность нашей оценки, если среднее всех доставок неизвестно?**

Продолжаем занятие 7: среднее меняется от выборки к выборке, а стандартная ошибка описывает этот разброс.
Теперь построим **интервал для неизвестного среднего**, затем научимся оценивать неопределённость
по одной выборке с помощью **бутстрапа**.

**План двух встреч:** первая — разделы 1–5 (смысл доверительного интервала и задача о доставках);
вторая — раздел 6 (бутстрап), обзор раздела 7 и практика 8. Дополнительные опыты можно оставить на семинар.
Перед проверкой гипотез разберём точность оценки. Статистические тесты и p-value будем изучать отдельно.

Перед опытом предскажите результат, после — объясните наблюдение. Важные выводы выделены рамками.
После интерактивов идут разобранные примеры NumPy и seaborn; задания для самостоятельной работы отмечены отдельно.
Все данные **синтетические**: известный генератор позволяет проверять наши методы.

В Colab: **«Среда выполнения → Выполнить все»**. Ноутбук содержит весь код интерактивов;
формулы загружаются через MathJax. Ползунки работают независимо от Python-ячеек:
чтобы повторить опыт в коде, измените числа в самой ячейке. Блоки «Дополнительно» можно пропускать.

In [ ]:
#@title Оформление и хелперы визуализации — запустите первой { display-mode: "form" }
from IPython.display import HTML, display

CSS = r"""
.vec-root{--bg:#12161c;--surface:#1a1f27;--ink:#e6e9ec;--soft:#a7b0ba;--line:#2a313b;
  --green:#5fd08a;--blue:#6ab0f3;--amber:#e0b25a;--rose:#e0757f;
  font-family:Georgia,'Times New Roman',serif;color:var(--ink);
  background:var(--bg);border-radius:14px;padding:2px 0;}
.vec-root .wrap{max-width:820px;margin:0 auto;padding:18px 22px;font-size:17px;line-height:1.65;}
.vec-root .wrap.wide{max-width:1180px;}
.vec-root h2{font-family:-apple-system,'Segoe UI',sans-serif;font-size:1.5rem;color:#fff;
  border-top:2px solid var(--line);padding-top:.5em;margin:1.4em 0 .5em;}
.vec-root h3{font-family:sans-serif;color:var(--green);font-size:1.15rem;margin:1.3em 0 .3em;}
.vec-root p{margin:.6em 0;} .vec-root b,.vec-root strong{color:#fff;}
.vec-root em{color:var(--soft);} .vec-root ul,.vec-root ol{padding-left:1.4em;} .vec-root li{margin:.28em 0;}
.vec-root .en{font-family:sans-serif;font-style:normal;color:#7f8a96;font-size:.88em;white-space:nowrap;}
.vec-root .where{font-family:sans-serif;font-size:.9rem;color:var(--soft);margin:.1em 0 .7em;}
.vec-root .where ul{margin:.3em 0;padding-left:1.3em;} .vec-root .where li{margin:.15em 0;}
.vec-root a{color:var(--green);}
.vec-root .box{border-radius:12px;padding:14px 18px;margin:1.2em 0;font-family:sans-serif;
  font-size:.95rem;border:1px solid var(--line);background:var(--surface);}
.vec-root .box .t{font-weight:700;font-size:.8rem;letter-spacing:.06em;text-transform:uppercase;margin-bottom:5px;}
.vec-root .box.def{background:#15251d;border-color:#27503a;} .vec-root .box.def .t{color:var(--green);}
.vec-root .box.idea{background:#14202e;border-color:#274a63;} .vec-root .box.idea .t{color:var(--blue);}
.vec-root .box.warn{background:#241f14;border-color:#5a4a27;} .vec-root .box.warn .t{color:var(--amber);}
.vec-root .box.take{background:#241419;border-color:#5a2730;} .vec-root .box.take .t{color:var(--rose);}
.vec-root .widget{border:1px solid var(--line);border-radius:12px;background:var(--surface);
  padding:16px;margin:1.4em 0;}
.vec-root .widget-title{font-family:sans-serif;font-weight:700;font-size:.95rem;margin-bottom:2px;color:#fff;}
.vec-root .widget-sub{font-family:sans-serif;font-size:.83rem;color:var(--soft);margin-bottom:10px;}
.vec-root .widget>svg{display:block;width:100%;height:auto;background:#0e1319;border-radius:10px;}
.vec-root .controls{display:flex;flex-wrap:wrap;gap:12px 20px;align-items:center;margin-top:12px;
  font-family:sans-serif;font-size:.88rem;color:var(--soft);}
.vec-root .control{display:flex;align-items:center;gap:8px;}
.vec-root input[type=range]{accent-color:var(--green);width:150px;}
.vec-root select{background:#0e1319;color:#e6e9ec;border:1px solid var(--line);border-radius:6px;padding:3px 7px;font-family:sans-serif;font-size:.85rem;}
.vec-root .val{font-family:'SF Mono',Menlo,monospace;font-size:.82rem;color:var(--green);min-width:40px;}
.vec-root .btn{font-family:sans-serif;font-size:.85rem;font-weight:600;border:1px solid var(--green);
  background:transparent;color:var(--green);padding:6px 14px;border-radius:8px;cursor:pointer;}
.vec-root .btn:hover{background:var(--green);color:#0e1319;}
.vec-root .readout{font-family:'SF Mono',Menlo,monospace;font-size:.84rem;background:#0e1319;
  padding:9px 12px;border-radius:8px;margin-top:12px;color:var(--soft);}
.vec-root .gridline{stroke:#20272f;stroke-width:1;} .vec-root .axis{stroke:#3a434f;stroke-width:1.2;}
.vec-root .tick-label{font-family:sans-serif;font-size:11px;fill:#6b7580;}
.vec-root code{font-family:Menlo,monospace;font-size:.86em;background:#0e1319;color:var(--amber);
  padding:1px 6px;border-radius:5px;}
.vec-root details{border:1px dashed #38414c;border-radius:10px;padding:4px 16px;margin:1.1em 0;background:#161b22;}
.vec-root summary{cursor:pointer;font-family:sans-serif;font-weight:700;color:var(--blue);padding:8px 0;}
.vec-root table{border-collapse:collapse;width:100%;margin:1em 0;font-family:sans-serif;font-size:.9rem;}
.vec-root th,.vec-root td{border:1px solid var(--line);padding:7px 10px;text-align:left;}
.vec-root td{color:#fff;}
.vec-root th{background:#0e1319;color:var(--green);font-weight:700;}
.vec-root tr:nth-child(even) td{background:#151a21;}
"""

JS = r"""
if(!window.__VEC_READY__){window.__VEC_READY__=true;
const NS="http://www.w3.org/2000/svg";
window.scale=(d,r)=>{const[a,b]=d,[c,e]=r;return v=>c+(v-a)*(e-c)/(b-a);};
window.el=(t,at={})=>{const x=document.createElementNS(NS,t);for(const k in at)x.setAttribute(k,at[k]);return x;};
window.clamp=(v,lo,hi)=>Math.max(lo,Math.min(hi,v));
window.arrow=(x1,y1,x2,y2,o={})=>{const g=el("g");const c=o.color||"#5fd08a",w=o.width||2.6;
  const ln={x1,y1,x2,y2,stroke:c,"stroke-width":w,"stroke-linecap":"round"};if(o.dash)ln["stroke-dasharray"]=o.dash;
  g.appendChild(el("line",ln));const L=Math.hypot(x2-x1,y2-y1);
  if(L>4){const an=Math.atan2(y2-y1,x2-x1),hl=o.head||12,ha=.42;
   const p1x=x2-hl*Math.cos(an-ha),p1y=y2-hl*Math.sin(an-ha),p2x=x2-hl*Math.cos(an+ha),p2y=y2-hl*Math.sin(an+ha);
   g.appendChild(el("polygon",{points:`${x2},${y2} ${p1x},${p1y} ${p2x},${p2y}`,fill:c}));}return g;};
window.svgPoint=(svg,ev)=>{const p=svg.createSVGPoint();const s=(ev.touches&&ev.touches.length)?ev.touches[0]:ev;
  p.x=s.clientX;p.y=s.clientY;return p.matrixTransform(svg.getScreenCTM().inverse());};
window.onDrag=(svg,h,cb)=>{let on=false;const st=e=>{on=true;e.preventDefault();};
  const mv=e=>{if(!on)return;e.preventDefault();cb(svgPoint(svg,e));};const en=()=>on=false;
  h.style.cursor="grab";h.addEventListener("mousedown",st);h.addEventListener("touchstart",st,{passive:false});
  window.addEventListener("mousemove",mv);window.addEventListener("touchmove",mv,{passive:false});
  window.addEventListener("mouseup",en);window.addEventListener("touchend",en);};
window.drawFrame=(g,c)=>{const{x,y,x0,x1,y0,y1,xt=[],yt=[]}=c;
  for(const tx of xt){g.appendChild(el("line",{x1:x(tx),y1:y(y0),x2:x(tx),y2:y(y1),class:"gridline"}));
    const t=el("text",{x:x(tx),y:y(y0)+15,class:"tick-label","text-anchor":"middle"});t.textContent=tx;g.appendChild(t);}
  for(const ty of yt){g.appendChild(el("line",{x1:x(x0),y1:y(ty),x2:x(x1),y2:y(ty),class:"gridline"}));
    const t=el("text",{x:x(x0)-8,y:y(ty)+4,class:"tick-label","text-anchor":"end"});t.textContent=ty;g.appendChild(t);}
  const ax=(y0<=0&&y1>=0)?0:y0,ay=(x0<=0&&x1>=0)?0:x0;
  g.appendChild(el("line",{x1:x(x0),y1:y(ax),x2:x(x1),y2:y(ax),class:"axis"}));
  g.appendChild(el("line",{x1:x(ay),y1:y(y0),x2:x(ay),y2:y(y1),class:"axis"}));};
const W=600,H=380,PL=36,PR=16,PT=16,PB=28;
window.X0=-6;window.X1=6;window.Y0=-3.7;window.Y1=3.7;
window.SX=scale([X0,X1],[PL,W-PR]);window.SY=scale([Y0,Y1],[H-PB,PT]);
window.IX=scale([PL,W-PR],[X0,X1]);window.IY=scale([H-PB,PT],[Y0,Y1]);
window.OX=SX(0);window.OY=SY(0);window.XT=[-6,-4,-2,2,4,6];window.YT=[-3,-1,1,3];
window.COL={a:"#6ab0f3",b:"#e0b25a",sum:"#e0757f",v:"#6ab0f3",res:"#5fd08a"};
window.freshFrame=(svg)=>{while(svg.firstChild)svg.removeChild(svg.firstChild);
  const g=el("g");svg.appendChild(g);drawFrame(g,{x:SX,y:SY,x0:X0,x1:X1,y0:Y0,y1:Y1,xt:XT,yt:YT});return g;};
window.handle=(g,dx,dy,color,onMove,svg)=>{const c=el("circle",{cx:SX(dx),cy:SY(dy),r:9,
  fill:color,"fill-opacity":.25,stroke:color,"stroke-width":2});g.appendChild(c);
  onDrag(svg,c,p=>onMove(clamp(IX(p.x),X0,X1),clamp(IY(p.y),Y0,Y1)));return c;};
window.label=(g,dx,dy,text,color)=>{const t=el("text",{x:SX(dx),y:SY(dy),fill:color,
  "font-family":"-apple-system,Segoe UI,sans-serif","font-size":15,"font-weight":700});
  t.textContent=text;g.appendChild(t);return t;};
// --- local SQUARE plane (equal px per unit): needed for honest circles/angles ---
window.plane=(half)=>{const aw=W-PL-PR,ah=H-PT-PB,ppu=Math.min(aw,ah)/(2*half);
  const cx=PL+aw/2,cy=PT+ah/2;
  const sx=v=>cx+v*ppu,sy=v=>cy-v*ppu,ix=px=>(px-cx)/ppu,iy=py=>(cy-py)/ppu;
  const ticks=[];for(let t=-Math.floor(half);t<=Math.floor(half);t++)if(t!==0)ticks.push(t);
  return {sx,sy,ix,iy,ox:sx(0),oy:sy(0),ppu,ticks,half,
    frame(g){drawFrame(g,{x:sx,y:sy,x0:-half,x1:half,y0:-half,y1:half,xt:ticks,yt:ticks});},
    drag(svg,g,dx,dy,color,cb){const c=el("circle",{cx:sx(dx),cy:sy(dy),r:9,fill:color,
      "fill-opacity":.25,stroke:color,"stroke-width":2});g.appendChild(c);
      onDrag(svg,c,p=>cb(clamp(ix(p.x),-half,half),clamp(iy(p.y),-half,half)));return c;}};};
}
"""

MJ_CFG = "<script>window.MathJax=window.MathJax||{tex:{inlineMath:[['$','$']],displayMath:[['$$','$$']]},svg:{fontCache:'global'}};</script>"
MJ = '<script src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-svg.js"></script>'

def viz(body, script="", wide=False):
    """Wrap an HTML body with the dark theme, JS helpers and MathJax. Self-contained (Colab).
    wide=True widens the content column (for full-width figures)."""
    head = "<style>" + CSS + "</style>"
    tail = ("<script>" + JS + "\n" + script +
            "\nif(window.MathJax&&MathJax.typesetPromise){MathJax.typesetPromise();}</script>")
    cls = "wrap wide" if wide else "wrap"
    return head + '<div class="vec-root"><div class="' + cls + '">' + body + "</div></div>" + MJ_CFG + MJ + tail

print("Оформление загружено. Функция viz() готова.")

D8_WIDGET_JS = '/* Standalone SVG simulations, embedded into 08_stat_ci_bootstrap.ipynb. */\n(function (global) {\n  \'use strict\';\n  const C = { blue:\'#6ab0f3\', green:\'#5fd08a\', gold:\'#e0b25a\', rose:\'#e0757f\', ink:\'#e6e9ec\', soft:\'#a7b0ba\', grid:\'#293440\' };\n  const fmt = (x, d=2) => (d ? Number(x).toFixed(d).replace(/\\.?0+$/, \'\') : String(Math.round(x))) || \'0\';\n  const mean = a => a.reduce((s,x)=>s+x,0)/a.length;\n  const sd = (a, ddof=0) => { const m=mean(a); return Math.sqrt(a.reduce((s,x)=>s+(x-m)**2,0)/(a.length-ddof)); };\n  const pdf = x => Math.exp(-x*x/2)/Math.sqrt(2*Math.PI);\n  function rng(seed) { return () => { seed|=0; seed=seed+0x6D2B79F5|0; let t=Math.imul(seed^seed>>>15,1|seed); t=t+Math.imul(t^t>>>7,61|t)^t; return ((t^t>>>14)>>>0)/4294967296; }; }\n  function normal(r) { return Math.sqrt(-2*Math.log(Math.max(r(),1e-12)))*Math.cos(2*Math.PI*r()); }\n  function draw(r, kind=\'skew\') {\n    if (kind===\'normal\') return normal(r);\n    if (kind===\'uniform\') return Math.sqrt(3)*(2*r()-1);\n    if (kind===\'rare\') { const y=r()<.01 ? 10+normal(r) : normal(r); return (y-.1)/Math.sqrt(1.99); }\n    return -Math.log(Math.max(r(),1e-12))-1;\n  }\n  const sample = (r,n,kind=\'skew\',mu=30,sigma=20) => Array.from({length:n},()=>mu+sigma*draw(r,kind));\n  function batch(r,n,B,kind=\'skew\',mu=30,sigma=20) {\n    const means=[], spreads=[];\n    for(let b=0;b<B;b++){ const a=sample(r,n,kind,mu,sigma); means.push(mean(a)); spreads.push(n>1?sd(a,1):NaN); }\n    return {means,spreads};\n  }\n  function el(tag, attrs={}, content=\'\') { const e=document.createElementNS(\'http://www.w3.org/2000/svg\',tag); for(const [k,v] of Object.entries(attrs))e.setAttribute(k,String(v)); if(content!==\'\')e.textContent=content; return e; }\n  function text(svg,x,y,t,color=C.soft,anchor=\'middle\',size=12){svg.appendChild(el(\'text\',{x,y,fill:color,\'text-anchor\':anchor,\'font-family\':\'system-ui,sans-serif\',\'font-size\':size},t));}\n  function frame(svg,xrange,yrange,xlabel,ylabel,title,showYTicks=true,xTicks=null,yTicks=null){\n    svg.replaceChildren();svg.setAttribute(\'viewBox\',\'0 0 640 360\');svg.setAttribute(\'role\',\'img\');svg.setAttribute(\'aria-label\',`${title}. ${xlabel}; ${ylabel}`);\n    svg.appendChild(el(\'title\',{},title));\n    const [lo,hi]=xrange,[bottom,top]=yrange,L=68,R=614,T=54,B=301;\n    const sx=x=>L+(x-lo)/(hi-lo)*(R-L),sy=y=>B-(y-bottom)/(top-bottom)*(B-T);\n    text(svg,L,22,title,C.ink,\'start\',14);text(svg,L,42,ylabel,C.soft,\'start\',11);\n    if(showYTicks)for(const y of (yTicks||Array.from({length:5},(_,i)=>bottom+(top-bottom)*i/4))){svg.appendChild(el(\'line\',{x1:L,x2:R,y1:sy(y),y2:sy(y),stroke:C.grid}));text(svg,L-8,sy(y)+4,fmt(y,3),C.soft,\'end\',11);}\n    for(const x of (xTicks||Array.from({length:6},(_,i)=>lo+(hi-lo)*i/5))){svg.appendChild(el(\'line\',{x1:sx(x),x2:sx(x),y1:T,y2:B,stroke:C.grid,\'stroke-opacity\':.5}));text(svg,sx(x),B+19,fmt(x),C.soft,\'middle\',11);}\n    text(svg,(L+R)/2,345,xlabel,C.ink,\'middle\',12);return {svg,lo,hi,bottom,top,L,R,T,B,sx,sy};\n  }\n  function line(p,coords,color=C.blue,width=2){p.svg.appendChild(el(\'path\',{d:coords.map(([x,y],i)=>`${i?\'L\':\'M\'} ${p.sx(x)} ${p.sy(y)}`).join(\' \'),fill:\'none\',stroke:color,\'stroke-width\':width}));}\n  function vline(p,x,label,color=C.gold,row=0){if(x<p.lo||x>p.hi)return;p.svg.appendChild(el(\'line\',{x1:p.sx(x),x2:p.sx(x),y1:p.T,y2:p.B,stroke:color,\'stroke-dasharray\':\'5 4\',\'stroke-width\':2}));text(p.svg,Math.max(p.L+55,Math.min(p.R-55,p.sx(x))),p.T+15+row*18,label,color);}\n  function hline(p,y,color=C.green){line(p,[[p.lo,y],[p.hi,y]],color);}\n  function dot(p,x,y,color=C.blue,r=3){p.svg.appendChild(el(\'circle\',{cx:p.sx(x),cy:p.sy(y),r,fill:color}));}\n  function bounds(a,pad=.08){const lo=Math.min(...a),hi=Math.max(...a),span=Math.max(hi-lo,1);return[lo-pad*span,hi+pad*span];}\n  function hist(svg,data,domain,xlabel,title,color=C.blue,normalCurve=null){\n    const [lo,hi]=domain,m=32,w=(hi-lo)/m,counts=Array(m).fill(0);let outside=0;\n    for(const x of data){if(x<lo||x>hi){outside++;continue;} counts[Math.min(m-1,Math.floor((x-lo)/w))]++;}\n    const heights=counts.map(c=>c/(data.length*w));\n    const ymax=Math.max(...heights,normalCurve?pdf(0)/normalCurve.sd:0,1e-6)*1.2;\n    const p=frame(svg,domain,[0,ymax],xlabel,\'Плотность (1 / единица горизонтальной оси)\',title);\n    counts.forEach((c,i)=>p.svg.appendChild(el(\'rect\',{x:p.sx(lo+i*w)+.3,y:p.sy(heights[i]),width:Math.max(0,p.sx(lo+(i+1)*w)-p.sx(lo+i*w)-.6),height:p.B-p.sy(heights[i]),fill:color,\'fill-opacity\':.6})));\n    if(normalCurve)line(p,Array.from({length:301},(_,i)=>{const x=lo+(hi-lo)*i/300;return[x,pdf((x-normalCurve.mu)/normalCurve.sd)/normalCurve.sd];}),C.gold,2.5);\n    if(outside)text(svg,p.R,p.T+14,`За окном: ${outside} из ${data.length}`,C.rose,\'end\',11);\n    return p;\n  }\n  const value=(root,k)=>root.querySelector(`[data-control="${k}"]`).value;\n  const num=(root,k)=>Number(value(root,k));\n  const chart=(root,k)=>root.querySelector(`[data-chart="${k}"]`);\n  const output=(root,s)=>{root.querySelector(\'[data-readout]\').innerHTML=s;};\n  function sync(root){root.querySelectorAll(\'input[type=range][data-control]\').forEach(e=>{const o=root.querySelector(`[data-value="${e.dataset.control}"]`);if(o)o.textContent=fmt(e.value);});}\n  function bind(root,render){root.querySelectorAll(\'[data-control]\').forEach(e=>e.addEventListener(\'input\',()=>{sync(root);render();}));sync(root);render();}\n  function button(root,k,fn){root.querySelector(`[data-action="${k}"]`).addEventListener(\'click\',fn);}\n\n  // Acklam\'s inverse-normal approximation; confidence controls use only 0.8..0.99.\n  function ppf(p) {\n    if (!(p > 0 && p < 1)) throw new Error(\'Probability must be between 0 and 1\');\n    const a=[-39.6968302866538,220.946098424521,-275.928510446969,138.357751867269,-30.6647980661472,2.50662827745924];\n    const b=[-54.4760987982241,161.585836858041,-155.698979859887,66.8013118877197,-13.2806815528857];\n    const c=[-.00778489400243029,-.322396458041136,-2.40075827716184,-2.54973253934373,4.37466414146497,2.93816398269878];\n    const d=[.00778469570904146,.32246712907004,2.445134137143,3.75440866190742];\n    if(p<.02425){const q=Math.sqrt(-2*Math.log(p));return (((((c[0]*q+c[1])*q+c[2])*q+c[3])*q+c[4])*q+c[5])/((((d[0]*q+d[1])*q+d[2])*q+d[3])*q+1);}\n    if(p>1-.02425)return -ppf(1-p);\n    const q=p-.5,r=q*q;return (((((a[0]*r+a[1])*r+a[2])*r+a[3])*r+a[4])*r+a[5])*q/(((((b[0]*r+b[1])*r+b[2])*r+b[3])*r+b[4])*r+1);\n  }\n  function quantile(a,q){const s=[...a].sort((x,y)=>x-y),t=(s.length-1)*q,i=Math.floor(t);return s[i]+(s[Math.min(i+1,s.length-1)]-s[i])*(t-i);}\n  const statistic=(a,kind=\'mean\')=>kind===\'median\'?quantile(a,.5):mean(a);\n  function resample(r,a){return Array.from({length:a.length},()=>a[Math.floor(r()*a.length)]);}\n  function boot(r,a,B,kind=\'mean\'){\n    const out=[];\n    for(let b=0;b<B;b++){\n      if(kind===\'mean\'){let total=0;for(let i=0;i<a.length;i++)total+=a[Math.floor(r()*a.length)];out.push(total/a.length);}\n      else out.push(statistic(resample(r,a),kind));\n    }\n    return out;\n  }\n  function interval(a,level=.95){const tail=(1-level)/2;return [quantile(a,tail),quantile(a,1-tail)];}\n  const covers=(ci,mu)=>ci[0]<=mu && mu<=ci[1];\n  function bandRect(p,low,high,color=C.green){\n    const left=Math.max(p.lo,low),right=Math.min(p.hi,high);\n    if(right>left)p.svg.appendChild(el(\'rect\',{x:p.sx(left),y:p.T,width:p.sx(right)-p.sx(left),height:p.B-p.T,fill:color,\'fill-opacity\':.12}));\n  }\n  function area(p,mu,se,lo,hi){\n    const pts=[[lo,0],...Array.from({length:151},(_,i)=>{const x=lo+(hi-lo)*i/150;return[x,pdf((x-mu)/se)/se];}),[hi,0]];\n    p.svg.appendChild(el(\'path\',{d:pts.map(([x,y],i)=>`${i?\'L\':\'M\'} ${p.sx(x)} ${p.sy(y)}`).join(\' \')+\' Z\',fill:C.gold,\'fill-opacity\':.25}));\n  }\n  function segment(p,lo,hi,y,color=C.green,point=null){\n    line(p,[[lo,y],[hi,y]],color,5);\n    for(const x of [lo,hi])p.svg.appendChild(el(\'line\',{x1:p.sx(x),x2:p.sx(x),y1:p.sy(y)-7,y2:p.sy(y)+7,stroke:color,\'stroke-width\':2}));\n    if(point!==null){dot(p,point,y,\'#0e1319\',6);dot(p,point,y,color,4);}\n  }\n  function forest(svg,means,intervals,truth,title,domain=null){\n    const k=Math.min(means.length,40),shown=intervals.slice(0,k);\n    const p=frame(svg,domain||bounds([truth,...shown.flat()]),[0,k+5],\'Значение среднего\',\'Номер исследования (первые \'+k+\')\',title,true,null,[1,10,20,30,40].filter(x=>x<=k));\n    for(let i=0;i<k;i++){\n      const ci=shown[i],color=covers(ci,truth)?C.green:C.rose,y=i+1;\n      line(p,[[ci[0],y],[ci[1],y]],color,2);dot(p,means[i],y,color,2.5);\n    }\n    vline(p,truth,\'Истинное среднее\',C.gold);\n    return p;\n  }\n  function singleInterval(svg,estimate,ci,truth,title,domain=null){\n    const p=frame(svg,domain||bounds([estimate,...ci,...(truth===null?[]:[truth])],.25),[0,1],\'Время, мин\',\'Точка — оценка; отрезок — интервал\',title,false);\n    const color=truth!==null&&!covers(ci,truth)?C.rose:C.green;\n    segment(p,ci[0],ci[1],.45,color,estimate);\n    text(svg,p.sx(ci[0]),p.sy(.45)+27,fmt(ci[0]),color);\n    text(svg,p.sx(ci[1]),p.sy(.45)+27,fmt(ci[1]),color);\n    text(svg,p.sx(estimate),p.sy(.45)-22,\'Оценка \'+fmt(estimate),color);\n    if(truth!==null)vline(p,truth,\'Истинное: \'+fmt(truth),C.gold);\n    return p;\n  }\n  function sampling(root){\n    let seed=101,history=[],last=[];\n    function add(B){const r=rng(seed++),n=num(root,\'n\');for(let i=0;i<B;i++){last=sample(r,n);history.push(mean(last));}history=history.slice(-3000);render();}\n    function render(){\n      const p=hist(chart(root,\'data\'),last,bounds(last),\'Время отдельной доставки, мин\',\'Одна выборка доставок\');\n      vline(p,30,\'Истинное среднее: 30\',C.gold);vline(p,mean(last),\'Среднее выборки: \'+fmt(mean(last)),C.green,1);\n      const q=hist(chart(root,\'means\'),history,bounds([20,40,...history]),\'Среднее целой выборки, мин\',\'Ответы по новым выборкам\',C.green);\n      vline(q,30,\'Истинное среднее: 30\',C.gold);\n      output(root,`<b>В выборке ${last.length} доставок; среднее ${fmt(mean(last))} мин. Получено ${history.length} выборок.</b><br>Слева одно число — отдельная доставка; справа одно число — среднее целой выборки. Все выборки получаем заново из одного и того же генератора. Истинное среднее остаётся 30 минут.`);\n      Object.assign(root.dataset,{n:last.length,B:history.length,estimate:mean(last)});\n    }\n    button(root,\'one\',()=>add(1));button(root,\'many\',()=>add(200));bind(root,()=>{history=[];add(1);});\n  }\n  function samplingBand(root){\n    let seed=201;\n    function render(){\n      const n=num(root,\'n\'),level=num(root,\'level\'),se=20/Math.sqrt(n),z=ppf((1+level)/2),h=z*se;\n      const means=batch(rng(seed),n,3000).means,domain=bounds([30-4*se,30+4*se,...means]);\n      const p=hist(chart(root,\'band\'),means,domain,\'Среднее выборки, мин\',\'Где оказываются выборочные средние\',C.blue,{mu:30,sd:se});\n      area(p,30,se,30-h,30+h);vline(p,30,\'μ = 30\',C.gold);\n      vline(p,30-h,fmt(30-h),C.green,1);vline(p,30+h,fmt(30+h),C.green,1);\n      const hit=means.filter(x=>Math.abs(x-30)<=h).length;\n      output(root,`<b>Границы: ${fmt(30-h)}–${fmt(30+h)} мин. SE = ${fmt(se)} мин.</b><br>Золотая кривая — нормальное приближение; под ней закрашено ${fmt(level*100)}%. По 3000 настоящим выборкам из генератора внутрь попало ${fmt(100*hit/3000)}% средних.<br>Для асимметричных доставок это приближение. При малом n форма гистограммы может заметно отличаться от кривой. Квантиль стандартной нормали: ${fmt(z,3)}.`);\n      Object.assign(root.dataset,{n,level,se,h,coverage:hit/3000});\n    }\n    button(root,\'new\',()=>{seed++;render();});bind(root,render);\n  }\n  function inversion(root){\n    let seed=301;\n    function render(){\n      const n=num(root,\'n\'),level=num(root,\'level\'),x=num(root,\'estimate\'),h=ppf((1+level)/2)*20/Math.sqrt(n),hit=Math.abs(x-30)<=h;\n      const domain=bounds([30-h,30+h,x-h,x+h],.14),color=hit?C.green:C.rose;\n      const p=frame(chart(root,\'fixed\'),domain,[0,1],\'Среднее, мин\',\'Фиксированный отрезок вокруг μ\',\'1. Где может оказаться среднее выборки?\',false);\n      segment(p,30-h,30+h,.45,C.gold,30);dot(p,x,.45,color,7);\n      text(p.svg,p.sx(30),p.sy(.45)-35,\'μ = 30\',C.gold);\n      text(p.svg,p.sx(x),p.sy(.45)+34,\'Среднее выборки: \'+fmt(x),color);\n      const q=frame(chart(root,\'moving\'),domain,[0,1],\'Среднее, мин\',\'Случайный отрезок вокруг среднего выборки\',\'2. Накрыл ли интервал истинное среднее?\',false);\n      segment(q,x-h,x+h,.45,color,x);vline(q,30,\'μ = 30\',C.gold);\n      text(q.svg,q.sx(x),q.sy(.45)+34,\'Среднее выборки: \'+fmt(x),color);\n      output(root,`<b>${hit?\'Попадание в обоих случаях\':\'Промах в обоих случаях\'}.</b> Расстояние от выборочного среднего до 30: ${fmt(Math.abs(x-30))} мин; половина ширины h = ${fmt(h)} мин.<br>Слева точка ${fmt(x)} ${hit?\'внутри\':\'вне\'} фиксированного отрезка. Справа число 30 ${hit?\'внутри\':\'вне\'} интервала [${fmt(x-h)}; ${fmt(x+h)}].<br>Оси одинаковы. Двигайте среднее выборки: граница попадания меняется одновременно на двух графиках. На первом графике случайна точка, на втором — границы интервала.`);\n      Object.assign(root.dataset,{n,level,estimate:x,h,fixedHit:hit,intervalHit:covers([x-h,x+h],30)});\n    }\n    button(root,\'new\',()=>{\n      const control=root.querySelector(\'[data-control="estimate"]\'),x=mean(sample(rng(seed++),num(root,\'n\')));\n      control.min=Math.min(+control.min,Math.floor(x));control.max=Math.max(+control.max,Math.ceil(x));control.value=x.toFixed(1);sync(root);render();\n    });bind(root,render);\n  }\n  function coverage(root){\n    let seed=401;\n    function render(){\n      const n=num(root,\'n\'),M=num(root,\'M\'),level=num(root,\'level\'),shape=value(root,\'shape\'),h=ppf((1+level)/2)*20/Math.sqrt(n);\n      const means=batch(rng(seed),n,M,shape).means,cis=means.map(x=>[x-h,x+h]),hit=cis.filter(ci=>covers(ci,30)).length;\n      forest(chart(root,\'forest\'),means,cis,30,\'Каждая строка — новое исследование\');\n      output(root,`<b>Накрыли истинное среднее: ${hit} из ${M} (${fmt(100*hit/M)}%). Заданный уровень: ${fmt(100*level)}%.</b><br>На графике первые ${Math.min(40,M)} исследований, счётчик учитывает все ${M}. Зелёный отрезок накрыл 30, розовый — не накрыл; точка — среднее соответствующей выборки. Ширина каждого интервала ${fmt(2*h)}.<br>${shape===\'normal\'?\'Для независимых нормальных наблюдений с известной σ метод имеет заданное покрытие. Частота в конечной серии всё равно колеблется. Нормальная модель здесь — условный показатель, не реальные времена доставок.\':\'Доставки асимметричны: используем ЦПТ и проверяем приближение. Малое n и конечное число исследований дают отличия от заданного уровня.\'}<br>Изменение уровня доверия сохраняет те же выборки: виден эффект одной лишь ширины интервалов.`);\n      Object.assign(root.dataset,{n,M,level,coverage:hit/M,width:2*h,shape});\n    }\n    button(root,\'new\',()=>{seed++;render();});bind(root,render);\n  }\n  const palette=[\'#6ab0f3\',\'#5fd08a\',\'#e0b25a\',\'#e0757f\',\'#bd9ce8\',\'#73d1d1\',\'#c8b365\',\'#df9dc7\',\'#89ad7a\',\'#dcaa80\',\'#8bace0\',\'#baaa92\'];\n  function cards(svg,indices,data,counts,title){\n    svg.replaceChildren();svg.setAttribute(\'viewBox\',\'0 0 640 360\');svg.setAttribute(\'role\',\'img\');svg.setAttribute(\'aria-label\',title);svg.appendChild(el(\'title\',{},title));\n    text(svg,24,26,title,C.ink,\'start\',15);\n    indices.forEach((idx,i)=>{\n      const x=22+(i%4)*154,y=48+Math.floor(i/4)*96,col=palette[idx%palette.length];\n      svg.appendChild(el(\'rect\',{x,y,width:140,height:82,rx:9,fill:col,\'fill-opacity\':.1,stroke:col,\'stroke-opacity\':.7}));\n      text(svg,x+12,y+20,\'№ \'+(idx+1),col,\'start\',12);\n      text(svg,x+70,y+45,fmt(data[idx],1)+\' мин\',C.ink,\'middle\',18);\n      text(svg,x+70,y+67,counts?\'Выбран \'+counts[idx]+\' раз\':\'Позиция \'+(i+1),C.soft,\'middle\',11);\n    });\n  }\n  function resampling(root){\n    let seed=501,data=sample(rng(seed),12),r=rng(701),history=[],last=[];\n    function add(B){\n      const replace=value(root,\'replace\')===\'yes\';\n      for(let b=0;b<B;b++){\n        last=Array.from({length:12},(_,i)=>i);\n        if(replace)last=last.map(()=>Math.floor(r()*12));\n        else for(let i=11;i>0;i--){const j=Math.floor(r()*(i+1));[last[i],last[j]]=[last[j],last[i]];}\n        history.push(mean(last.map(i=>data[i])));\n      }\n      history=history.slice(-5000);render();\n    }\n    function render(){\n      const counts=Array(12).fill(0);last.forEach(i=>counts[i]++);\n      cards(chart(root,\'original\'),Array.from({length:12},(_,i)=>i),data,counts,\'Исходные 12 наблюдений\');\n      cards(chart(root,\'resample\'),last,data,null,\'Последняя повторная выборка: 12 записей\');\n      const p=hist(chart(root,\'bootmeans\'),history,bounds([mean(data)-8,mean(data)+8,...history]),\'Среднее повторной выборки, мин\',\'Средние из одной исходной выборки\',C.green);\n      vline(p,mean(data),\'Исходное среднее: \'+fmt(mean(data)),C.gold);\n      output(root,`<b>Повторений: ${history.length}. Разных исходных записей в последней выборке: ${new Set(last).size} из 12.</b><br>Исходное среднее: ${fmt(mean(data))}; последнее повторное: ${fmt(mean(last.map(i=>data[i])))} мин.<br>${value(root,\'replace\')===\'yes\'?\'Одинаковые номера и цвета обозначают одну исходную запись. Некоторые записи повторяются; размер каждой выборки остаётся 12. Это обычный бутстрап.\':\'Без возвращения мы взяли все 12 записей в другом порядке. Среднее осталось прежним. Это сравнение показывает, зачем бутстрапу возвращение.\'}<br>Новые реальные наблюдения здесь не появляются.`);\n      Object.assign(root.dataset,{B:history.length,n:12,unique:new Set(last).size,spread:history.length>1?sd(history,1):0,estimate:mean(data)});\n    }\n    button(root,\'one\',()=>add(1));button(root,\'many\',()=>add(200));\n    button(root,\'source\',()=>{data=sample(rng(++seed),12);history=[];r=rng(seed+200);add(1);});\n    bind(root,()=>{history=[];r=rng(seed+200);add(1);});\n  }\n  function bootstrap(root){\n    let seed=601,bootSeed=801;\n    function render(){\n      const n=num(root,\'n\'),B=num(root,\'B\'),kind=value(root,\'stat\'),level=num(root,\'level\'),show=value(root,\'truth\')===\'show\';\n      const data=sample(rng(seed),n),estimate=statistic(data,kind),truth=kind===\'mean\'?30:10+20*Math.log(2);\n      const reps=boot(rng(bootSeed),data,B,kind),ci=interval(reps,level),se=sd(reps,1),name=kind===\'mean\'?\'Среднее\':\'Медиана\';\n      const p=hist(chart(root,\'data\'),data,bounds(data),\'Время отдельной доставки, мин\',\'Исходная выборка: \'+n+\' доставок\');\n      vline(p,estimate,name+\' выборки: \'+fmt(estimate),C.green);\n      const q=hist(chart(root,\'distribution\'),reps,bounds([...reps,estimate,...(show?[truth]:[])]),name+\' повторной выборки, мин\',\'Бутстрап-распределение: \'+B+\' повторений\',C.green);\n      bandRect(q,...ci);vline(q,estimate,\'Исходная оценка\',C.green);\n      if(show)vline(q,truth,\'Истинное значение\',C.gold,1);\n      singleInterval(chart(root,\'interval\'),estimate,ci,show?truth:null,\'Интервал по квантилям бутстрап-распределения\');\n      output(root,`<b>${name}: ${fmt(estimate)} мин. Интервал: [${fmt(ci[0])}; ${fmt(ci[1])}] мин. Бутстрап-SE: ${fmt(se)} мин.</b><br>n = ${n} исходных наблюдений; B = ${B} повторных выборок. Зелёная полоса на гистограмме — центральные ${fmt(level*100)}% бутстрап-значений, отрезок внизу — их границы на исходной шкале.<br>${show?`Для проверки открыто истинное значение: ${fmt(truth)} мин. Этот интервал его ${covers(ci,truth)?\'накрыл\':\'не накрыл\'}.`:\'Истинное значение скрыто: при расчёте интервала мы его не использовали.\'}<br>Кнопка «Повторить бутстрап» сохраняет исходные данные; кнопка «Новая исходная выборка» собирает новые доставки. Увеличение B уточняет вычисление, но не увеличивает n. ${kind===\'median\'?\'Для медианы не используем формулу SE среднего.\':\'Для среднего оценка s / √n по этой выборке: \'+fmt(sd(data,1)/Math.sqrt(n))+\' мин.\'}`);\n      Object.assign(root.dataset,{n,B,estimate,truth,lower:ci[0],upper:ci[1],se,analyticSE:sd(data,1)/Math.sqrt(n),kind});\n    }\n    button(root,\'source\',()=>{seed++;render();});button(root,\'repeat\',()=>{bootSeed++;render();});bind(root,render);\n  }\n  function bootcoverage(root){\n    let seed=901;\n    function render(){\n      const n=num(root,\'n\'),M=num(root,\'M\'),B=500,r=rng(seed),rboot=rng(seed+10000),means=[],cis=[],known=[],h=ppf(.975)*20/Math.sqrt(n);\n      for(let i=0;i<M;i++){const data=sample(r,n),m=mean(data);means.push(m);cis.push(interval(boot(rboot,data,B)));known.push([m-h,m+h]);}\n      const domain=bounds([30,...cis.slice(0,40).flat(),...known.slice(0,40).flat()]);\n      forest(chart(root,\'known\'),means,known,30,\'Нормальный интервал: σ известна\',domain);\n      forest(chart(root,\'bootstrap\'),means,cis,30,\'Бутстрап: интервал по квантилям\',domain);\n      const hit=cis.filter(ci=>covers(ci,30)).length,hitKnown=known.filter(ci=>covers(ci,30)).length;\n      output(root,`<b>Из ${M} новых исследований: формула накрыла 30 в ${hitKnown} (${fmt(100*hitKnown/M)}%); бутстрап — в ${hit} (${fmt(100*hit/M)}%).</b><br>Обе панели используют одни и те же исходные выборки. На каждой показаны первые ${Math.min(40,M)} исследований. Внутри каждого исследования B = ${B} бутстрап-повторений по n = ${n} доставок.<br>Только метод слева получает истинную σ = 20. Справа используются одни наблюдения. Это сравнение условий, а не соревнование равноправных методов.<br>Для асимметричного источника и небольших выборок квантильный интервал может накрывать истину реже заявленных 95%. Частоты в конечной серии и сами оценки квантилей тоже случайно колеблются.`);\n      Object.assign(root.dataset,{n,M,B,coverage:hit/M,knownCoverage:hitKnown/M});\n    }\n    button(root,\'new\',()=>{seed++;render();});bind(root,render);\n  }\n  function model(root){\n    let seed=1101;\n    function render(){\n      const data=sample(rng(seed),60),kind=value(root,\'kind\'),r=rng(seed+1000),m=mean(data),s=sd(data),B=1000;\n      const draws=kind===\'empirical\'?Array.from({length:3000},()=>data[Math.floor(r()*data.length)]):Array.from({length:3000},()=>m+s*normal(r));\n      const domain=bounds([...data,...draws]);\n      const p=hist(chart(root,\'observed\'),data,domain,\'Время, мин\',\'Данные: 60 доставок\');vline(p,m,\'Выборочное среднее\',C.green);\n      const q=hist(chart(root,\'generated\'),draws,domain,\'Сгенерированное время, мин\',kind===\'empirical\'?\'Повторные значения из наблюдений\':\'Значения из подобранной нормальной модели\');vline(q,m,\'Среднее исходных данных\',C.green);\n      const reps=kind===\'empirical\'?boot(rng(seed+2000),data,B):batch(rng(seed+2000),data.length,B,\'normal\',m,s).means;\n      const t=hist(chart(root,\'statistics\'),reps,bounds(reps),\'Среднее повторной выборки, мин\',\'В обоих случаях пересчитываем среднее\',C.green);vline(t,m,\'Исходное среднее\',C.gold);\n      output(root,`<b>${kind===\'empirical\'?\'Непараметрический\':\'Параметрический\'} бутстрап.</b> Справа вверху 3000 отдельных сгенерированных значений; внизу ${B} средних выборок по 60 значений.<br>${kind===\'empirical\'?\'Каждое значение справа уже встречалось в исходных данных. Мы используем эмпирическое распределение.\':\'Мы предположили нормальное распределение и оценили его центр и разброс по данным. Возможны новые значения. Отрицательных времён сгенерировано: \'+draws.filter(x=>x<0).length+\'. Для доставок это показывает возможную проблему выбранной модели; параметрический бутстрап наследует её предположения.\'}<br>Схему выбирают по устройству задачи и обоснованности модели, а не по тому, какой интервал оказался уже.`);\n      Object.assign(root.dataset,{kind,n:60,negative:draws.filter(x=>x<0).length,estimate:m,se:sd(reps,1)});\n    }\n    button(root,\'new\',()=>{seed++;render();});bind(root,render);\n  }\n  function clusters(root){\n    let seed=1201;\n    function render(){\n      const D=num(root,\'groups\'),m=8,tau=num(root,\'tau\'),r=rng(seed),groups=[];\n      for(let d=0;d<D;d++){const shift=tau*normal(r);groups.push(Array.from({length:m},()=>30+shift+2*normal(r)));}\n      const flat=groups.flat(),groupMeans=groups.map(mean),rows=boot(rng(seed+1000),flat,2000),whole=boot(rng(seed+2000),groupMeans,2000);\n      const ticks=Array.from({length:Math.min(6,D)},(_,i)=>Math.round(1+i*(D-1)/(Math.min(6,D)-1)));\n      const p=frame(chart(root,\'groups\'),[.5,D+.5],bounds(flat),\'Номер дня\',\'Условный показатель\',\'Внутри дня есть общий случайный сдвиг\',true,ticks);\n      groups.forEach((g,i)=>g.forEach((x,j)=>dot(p,i+1+(j-3.5)*.07,x,palette[i%12],3)));hline(p,mean(flat),C.gold);\n      const domain=bounds([...rows,...whole]);\n      const q=hist(chart(root,\'rows\'),rows,domain,\'Общее среднее, усл. ед.\',\'Бутстрап отдельных строк\',C.blue);vline(q,mean(flat),\'Исходное среднее\',C.gold);\n      const t=hist(chart(root,\'whole\'),whole,domain,\'Общее среднее, усл. ед.\',\'Бутстрап целых дней\',C.green);vline(t,mean(flat),\'Исходное среднее\',C.gold);\n      output(root,`<b>${D} независимых дней по ${m} записей.</b> SE при пересэмплировании строк: ${fmt(sd(rows,1))}; при пересэмплировании дней: ${fmt(sd(whole,1))}.<br>В каждом повторении справа выбираем с возвращением ${D} дней и берём все 8 записей каждого выбранного дня. Повторившийся день повторяется целиком.<br>При заметном общем сдвиге перемешивание отдельных строк разрушает зависимость и обычно занижает неопределённость. Здесь дни независимы и одинакового размера; для последовательных зависимых дней нужна другая схема. Небольшое число дней ограничивает надёжность и кластерного бутстрапа.<br>Это синтетический пример устройства данных, а не расчёт по реальным автобусам.`);\n      Object.assign(root.dataset,{groups:D,n:D*m,rowSE:sd(rows,1),clusterSE:sd(whole,1)});\n    }\n    button(root,\'new\',()=>{seed++;render();});bind(root,render);\n  }\n  const widgets={sampling,band:samplingBand,inversion,coverage,resampling,bootstrap,bootcoverage,model,clusters};\n  global.InferenceLesson8={mount:(root,kind)=>{if(!root||!widgets[kind])throw new Error(\'Unknown widget: \'+kind);widgets[kind](root);},math:{rng,normal,sample,batch,mean,sd,ppf,quantile,resample,boot,interval,covers}};\n})(typeof window!==\'undefined\'?window:globalThis);\n'

import json
import uuid
import re
from html import escape

D8_CSS = r"""
.vec-root .d8-controls{display:flex;flex-wrap:wrap;gap:12px 24px;margin:16px 0 22px;font-family:system-ui,sans-serif;}
.vec-root .d8-control{display:flex;align-items:center;flex-wrap:wrap;gap:8px;color:var(--ink);font-size:14px;min-height:36px;}
.vec-root .d8-control input[type=range]{width:155px;min-height:30px;}
.vec-root .d8-control output{font-variant-numeric:tabular-nums;color:var(--green);min-width:42px;}
.vec-root .d8-control select{max-width:100%;padding:8px;}
.vec-root .d8-control input:disabled{opacity:.3;}
.vec-root .d8-charts{display:grid;grid-template-columns:repeat(auto-fit,minmax(min(100%,380px),1fr));gap:16px;}
.vec-root .d8-charts svg{display:block;width:100%;height:auto;background:#0e1319;border-radius:10px;}
.vec-root .d8-charts svg:last-child:nth-child(3){grid-column:1/-1;max-width:780px;justify-self:center;}
.vec-root[data-widget=clusters] .d8-charts svg:first-child{grid-column:1/-1;max-width:780px;justify-self:center;}
.vec-root[data-widget=clusters] .d8-charts svg:last-child{grid-column:auto;max-width:none;}
.vec-root .readout{overflow-wrap:anywhere;font:15px/1.75 system-ui,sans-serif;}
.vec-root .d8-question{font-family:system-ui,sans-serif;font-size:14px;color:var(--ink);}
.vec-root .d8-footnote{font-family:system-ui,sans-serif;font-size:12px;color:var(--soft);}
.vec-root .d8-equation{overflow-x:auto;overflow-y:hidden;padding:5px 0;}
.vec-root .where{line-height:1.7;}
.vec-root .d8-table{overflow-x:auto;}
.vec-root input:focus-visible,.vec-root select:focus-visible,.vec-root button:focus-visible{outline:2px solid #e0b25a;outline-offset:4px;}
@media(max-width:600px){.vec-root .wrap{padding:14px;font-size:16px;}.vec-root .widget{padding:10px;}.vec-root .d8-control{width:100%;}}
"""

def lesson8_viz(body, widget=None):
    """Scope controls to a fresh root so rerunning cells keeps widgets independent."""
    # HTML is parsed before MathJax. Protect comparisons and ampersands in both
    # display equations and inline symbol glossaries without escaping HTML tags.
    body = re.sub(
        r"(?<!\\)\$\$.*?(?<!\\)\$\$|(?<!\\)\$.*?(?<!\\)\$",
        lambda match: escape(match.group(0), quote=False),
        body,
        flags=re.DOTALL,
    )
    root_id = "d8_" + uuid.uuid4().hex
    script = D8_WIDGET_JS + "\n" + r"""
    (function(){
      const root = document.getElementById(__ROOT__);
      const widget = __WIDGET__;
      if (widget) window.InferenceLesson8.mount(root, widget);
      const typeset = () => window.MathJax.typesetPromise([root]);
      if (!window.__D8_MATH_READY__) {
        if (window.MathJax && window.MathJax.typesetPromise) {
          window.__D8_MATH_READY__ = window.MathJax.startup.promise;
        } else {
          window.MathJax = {tex:{inlineMath:[['$','$']],displayMath:[['$$','$$']]},svg:{fontCache:'local'},startup:{typeset:false}};
          window.__D8_MATH_READY__ = new Promise((resolve, reject) => {
            const loader = document.createElement('script');
            loader.src = 'https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-svg.js';
            loader.onload = () => window.MathJax.startup.promise.then(resolve, reject);
            loader.onerror = reject;
            document.head.appendChild(loader);
          });
        }
      }
      window.__D8_MATH_READY__ = window.__D8_MATH_READY__.then(typeset).catch(() => {
        if(root.querySelector('.d8-equation')) {
          const notice = document.createElement('p');
          notice.textContent = 'Для отображения формул подключитесь к интернету и перезапустите ячейку. Графики работают без загрузки MathJax.';
          root.appendChild(notice);
        }
      });
    })();
    """.replace("__ROOT__", json.dumps(root_id)).replace("__WIDGET__", json.dumps(widget))
    return ("<style>" + CSS + D8_CSS + "</style>"
            + '<div class="vec-root" data-widget="' + (widget or "theory") + '" id="' + root_id + '"><div class="wrap wide">'
            + body + '</div></div><script>' + script + '</script>')

print("Занятие 8 готово: графики работают в браузере; формулы загружаются через MathJax.")

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import norm, bootstrap

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False,
                     "axes.spines.right": False})
rng = np.random.default_rng(2026)

## 1. Среднее посчитали. Насколько оно точное?

Изучаем доставки одного процесса при неизменных условиях. В нашей модели каждое время независимо
от других, а распределение времён одно и то же. Одна выборка содержит 100 доставок.

Нас интересует **среднее время всех доставок этого процесса**. Сначала мы знаем ответ из генератора
и проверяем расчёты. Позже скроем его и будем использовать только выборку.

In [ ]:
#@title Интерактив — Одна выборка — один ответ { display-mode: "form" }
display(HTML(lesson8_viz('<div class="widget"><div class="widget-title">Одна выборка — один ответ</div><p class="widget-sub">Предскажите, изменится ли среднее. Получите одну новую выборку, затем ещё 200. Сравните левую и правую оси.</p><div class="d8-controls"><label class="d8-control"><span>Доставок в выборке</span><select data-control="n"><option value="100">100</option><option value="20">20</option><option value="400">400</option></select></label><button class="btn" data-action="one">Новая выборка</button><button class="btn" data-action="many">Ещё 200 выборок</button></div><div class="d8-charts"><svg data-chart="data"></svg><svg data-chart="means"></svg></div><div class="readout" data-readout role="status" aria-live="polite"></div><p class="d8-question"><b>Проверь себя.</b> Почему столбцы справа описывают средние, а не отдельные доставки?</p><p class="d8-footnote">Учебная симуляция. Числа округлены; новое повторение может дать другой результат.</p></div>', widget='sampling')))

In [ ]:
#@title Оценка и её точность { display-mode: "form" }
display(HTML(lesson8_viz('<div class="box def"><div class="t">Точечная оценка · point estimate</div><p>Число, рассчитанное по выборке для приближения неизвестного параметра генеральной совокупности. Например, выборочное среднее оценивает среднее ГС.</p></div><div class="box take"><div class="t">Что предстоит добавить</div><p>Одного среднего недостаточно для описания точности. Добавим интервал, способ построения которого учитывает случайность выборки.</p></div><div class="box idea"><div class="t">Два разных вопроса</div><p><b>Каково среднее всех доставок?</b> — вопрос об оценке параметра.<br><b>Сколько займёт следующая доставка?</b> — вопрос об отдельном наблюдении. В этом занятии строим интервал для среднего.</p></div>')))

**Разобранный пример: получаем выборку и строим гистограмму.**

`rng.exponential(scale=20, size=n)` создаёт `n` независимых положительных времён ожидания
из экспоненциального распределения — распределения с длинным правым хвостом, у которого здесь
среднее и SD равны 20. Добавление 10 минут даёт модель из занятия 7: среднее 30, SD 20.
Изучать формулу этого распределения сейчас не требуется: это наш известный учебный генератор.

Повторный запуск ячейки даст новые доставки. `mu_true` и `sigma_true` — сведения о генераторе;
в реальном исследовании такие сведения могут быть недоступны.

In [ ]:
n = 100
mu_true = 30.0
sigma_true = 20.0
delivery_times = 10 + rng.exponential(scale=20, size=n)
sample_mean = delivery_times.mean()

ax = sns.histplot(delivery_times, bins=20, color="#478ac9")
ax.axvline(sample_mean, color="#14804a", lw=2, label="Среднее выборки")
ax.axvline(mu_true, color="#b67a10", ls="--", lw=2, label="Истинное среднее")
ax.set(xlabel="Время одной доставки, мин", ylabel="Число доставок",
       title=f"{n} наблюдений; среднее выборки {sample_mean:.2f} мин")
ax.legend()
plt.show()

## 2. Где окажутся средние новых выборок?

При повторном сборе данных меняется выборочное среднее. **Стандартная ошибка (SE)** — стандартное
отклонение распределения этих средних. Вспомним её смысл из занятия 7.

Теперь выделим область, в которую среднее выборки должно попадать примерно в 95% повторений.
Это пока **фиксированная область вокруг известного среднего ГС**.

In [ ]:
#@title Интерактив — 95% под кривой распределения средних { display-mode: "form" }
display(HTML(lesson8_viz('<div class="widget"><div class="widget-title">95% под кривой распределения средних</div><p class="widget-sub">Начните со 100 доставок. Уменьшите n до 5, затем увеличьте до 400. Сравните заданную площадь и фактическую долю попаданий.</p><div class="d8-controls"><label class="d8-control"><span>Размер выборки n</span><select data-control="n"><option value="100">100</option><option value="5">5</option><option value="20">20</option><option value="400">400</option></select></label><label class="d8-control"><span>Центральная область</span><select data-control="level"><option value="0.95">95%</option><option value="0.8">80%</option><option value="0.9">90%</option><option value="0.99">99%</option></select></label><button class="btn" data-action="new">Новые 3000 выборок</button></div><div class="d8-charts"><svg data-chart="band"></svg></div><div class="readout" data-readout role="status" aria-live="polite"></div><p class="d8-question"><b>Проверь себя.</b> Почему число 95% относится здесь к выборочным средним, а не к отдельным доставкам?</p><p class="d8-footnote">Учебная симуляция. Числа округлены; новое повторение может дать другой результат.</p></div>', widget='band')))

In [ ]:
#@title От SE к границам области { display-mode: "form" }
display(HTML(lesson8_viz('\n<p>Если распределение выборочного среднего близко к нормальному, центральная область шириной\nпримерно четыре SE содержит около 95% средних. Точный коэффициент для каждой стороны — примерно 1,96.</p><div class=\'d8-equation\'>$$SE(\\bar X)=\\frac{\\sigma}{\\sqrt n},\\qquad h=1.96\\frac{\\sigma}{\\sqrt n}.$$</div><div class=\'where\'><b>Обозначения.</b> $\\bar X$ — среднее случайной выборки; $SE$ — его стандартная ошибка; $\\sigma$ — известное SD ГС; $n$ — размер выборки; $h$ — расстояние от центра до каждой границы. Число 1,96 получено из стандартного нормального распределения.</div><div class=\'d8-equation\'>$$P\\!\\left(\\mu-h\\leq\\bar X\\leq\\mu+h\\right)\\approx0.95.$$</div><div class=\'where\'><b>Обозначения.</b> $P$ — вероятность; $\\mu$ — истинное среднее ГС; $\\bar X$ — случайное выборочное среднее; $h=1.96\\sigma/\\sqrt n$ — половина ширины области; $\\sigma$ — SD ГС; $n$ — размер выборки.</div><div class="box warn"><div class="t">Условия имеют значение</div><p>Формула SE требует независимых одинаково распределённых наблюдений с конечной дисперсией. Нормальная форма средних точна для нормальной ГС; для наших асимметричных доставок используем приближение по ЦПТ. Единого правила «n ≥ 30 всегда достаточно» нет.</p></div><details><summary>Откуда 1,96 и как получить другой уровень?</summary>\n<p>Квантиль — значение, слева от которого лежит заданная доля распределения. Для центральных 95%\nостаётся по 2,5% в каждом хвосте: нужна квантиль уровня 97,5%. Функция <code>norm.ppf</code>\nнаходит квантиль стандартной нормали. Это обратная операция к знакомой <code>norm.cdf</code>.</p><div class=\'d8-equation\'>$$z=\\Phi^{-1}\\!\\left(\\frac{1+c}{2}\\right),\\qquad h=z\\frac{\\sigma}{\\sqrt n}.$$</div><div class=\'where\'><b>Обозначения.</b> $c$ — выбранная центральная вероятность (например, 0,95); $\\Phi$ — CDF стандартной нормали; $\\Phi^{-1}$ — её функция квантилей; $z$ — нужная квантиль; $h$ — половина ширины; $\\sigma$ — известное SD ГС; $n$ — размер выборки.</div></details><p>Источник: <a href="https://www.itl.nist.gov/div898/handbook/prc/section1/prc14.htm">NIST — доверительные интервалы при известной σ</a>.</p>')))

**Разобранный пример: повторяем сбор данных в цикле.** В списке `sample_means` одна запись — среднее
целой выборки. `norm.ppf(0.975)` даёт коэффициент без округления до 1,96.
Гистограмма оценивает плотность средних, поэтому её можно сравнивать с нормальной плотностью на той же шкале.

In [ ]:
n = 100
repetitions = 3000
sample_means = []
for _ in range(repetitions):
    new_sample = 10 + rng.exponential(scale=20, size=n)
    sample_means.append(new_sample.mean())
sample_means = np.array(sample_means)

se_known = sigma_true / np.sqrt(n)
z = norm.ppf(0.975)
h = z * se_known
x_grid = np.linspace(sample_means.min(), sample_means.max(), 400)
density = norm.pdf(x_grid, loc=mu_true, scale=se_known)

ax = sns.histplot(sample_means, bins=35, stat="density", color="#478ac9")
ax.plot(x_grid, density, color="#b67a10", label="Нормальное приближение")
inside = (x_grid >= mu_true - h) & (x_grid <= mu_true + h)
ax.fill_between(x_grid, 0, density, where=inside, color="#e0b25a", alpha=0.4)
ax.axvline(mu_true, color="#b67a10", ls="--", label="Среднее ГС")
ax.set(xlabel="Среднее одной выборки, мин", ylabel="Плотность, 1/мин")
ax.legend()
plt.show()
print("Доля средних внутри области:", np.mean(np.abs(sample_means - mu_true) <= h))

## 3. Истинное среднее неизвестно: меняем центр интервала

Предыдущая область требовала знания среднего ГС. В исследовании именно его и нужно найти.
Но выражение «два числа отличаются не больше чем на h» одинаково в обе стороны.
Используем это, чтобы построить интервал **вокруг наблюдаемого среднего**.

In [ ]:
#@title Интерактив — Одно событие — два способа его показать { display-mode: "form" }
display(HTML(lesson8_viz('<div class="widget"><div class="widget-title">Одно событие — два способа его показать</div><p class="widget-sub">Двигайте среднее выборки. Найдите момент, когда попадание сменится промахом. Затем измените размер выборки и уровень доверия.</p><div class="d8-controls"><label class="d8-control"><span>Среднее выборки, мин</span><input type="range" data-control="estimate" min="15" max="60" step="0.1" value="33"><output data-value="estimate">33</output></label><label class="d8-control"><span>Размер выборки</span><select data-control="n"><option value="100">100</option><option value="25">25</option><option value="400">400</option></select></label><label class="d8-control"><span>Уровень</span><select data-control="level"><option value="0.95">95%</option><option value="0.8">80%</option><option value="0.99">99%</option></select></label><button class="btn" data-action="new">Среднее новой случайной выборки</button></div><div class="d8-charts"><svg data-chart="fixed"></svg><svg data-chart="moving"></svg></div><div class="readout" data-readout role="status" aria-live="polite"></div><p class="d8-question"><b>Проверь себя.</b> Что случайно на левом графике? Что случайно на правом? Почему цвет меняется одновременно?</p><p class="d8-footnote">Учебная симуляция. Числа округлены; новое повторение может дать другой результат.</p></div>', widget='inversion')))

In [ ]:
#@title Почему вероятность сохраняется { display-mode: "form" }
display(HTML(lesson8_viz('<div class=\'d8-equation\'>$$\\mu-h\\leq\\bar X\\leq\\mu+h\\quad\\Longleftrightarrow\\quad\\bar X-h\\leq\\mu\\leq\\bar X+h.$$</div><div class=\'where\'><b>Обозначения.</b> $\\mu$ — фиксированное среднее ГС; $\\bar X$ — случайное выборочное среднее; $h$ — одна и та же половина ширины. $\\Longleftrightarrow$ означает равносильность: условия выполняются одновременно.</div><div class="box idea"><div class="t">Меняется то, что мы считаем неизвестным</div><p>Слева неподвижный интервал и случайная точка. Справа неподвижное среднее ГС и случайный интервал. Это одно событие попадания.</p></div><div class="box def"><div class="t">Доверительный интервал · confidence interval (CI)</div><p>Интервал, границы которого вычисляют по выборке по правилу с заданной вероятностью покрытия неизвестного параметра. При повторном сборе данных это правило должно накрывать параметр с указанной вероятностью; для приближённых методов — приблизительно.</p></div><div class=\'d8-equation\'>$$I(\\bar X)=[\\bar X-h,\\ \\bar X+h],\\qquad P\\!\\left(\\mu\\in I(\\bar X)\\right)\\approx0.95.$$</div><div class=\'where\'><b>Обозначения.</b> $I(\\bar X)$ — случайный интервал по выборке; $\\bar X$ — её среднее; $h=1.96\\sigma/\\sqrt n$; $\\sigma$ — известное SD ГС; $n$ — размер выборки; $\\mu$ — среднее ГС; $P$ — вероятность; $\\in$ означает «принадлежит».</div><div class="box take"><div class="t">Формулировка для устного ответа</div><p>95%-й доверительный интервал строят способом, который при многократном повторении исследования накрывает истинный параметр примерно в 95% случаев.</p></div><div class="box warn"><div class="t">После вычисления границ</div><p>У конкретного рассчитанного интервала границы уже фиксированы. Он либо накрыл фиксированное среднее ГС, либо нет; мы обычно не знаем, какой случай произошёл. 95% характеризуют процедуру построения. Само выборочное среднее находится в центре нашего симметричного интервала всегда.</p></div>')))

**Разобранный пример: один интервал на обычном графике.** Используем `delivery_times` из раздела 1.
Функции `hlines` и `scatter` из Matplotlib рисуют отрезок и точку; seaborn задаёт оформление.
Это интервал из явно рассчитанных границ, без автоматического расчёта библиотекой.

In [ ]:
estimate = delivery_times.mean()
margin = norm.ppf(0.975) * sigma_true / np.sqrt(len(delivery_times))
ci_low, ci_high = estimate - margin, estimate + margin

fig, ax = plt.subplots(figsize=(10, 3))
ax.hlines(y=0, xmin=ci_low, xmax=ci_high, color="#14804a", lw=5)
ax.scatter([estimate], [0], color="#14804a", s=80, zorder=3, label="Среднее выборки")
ax.axvline(mu_true, color="#b67a10", ls="--", label="Среднее ГС: известно для проверки")
ax.set(xlabel="Среднее время доставки, мин", yticks=[], ylim=(-1, 1),
       title=f"Приближённый 95%-й ДИ: [{ci_low:.2f}; {ci_high:.2f}] мин")
ax.legend()
plt.show()

## 4. 95% интервалов: повторяем всё исследование

Один интервал не показывает надёжность метода. Для проверки нужно каждый раз получать **новые исходные данные**.
В симуляции мы можем это сделать и подсчитать долю попаданий. В реальном исследовании истинный ответ обычно скрыт.

In [ ]:
#@title Интерактив — Истинное среднее неподвижно, интервалы меняются { display-mode: "form" }
display(HTML(lesson8_viz('<div class="widget"><div class="widget-title">Истинное среднее неподвижно, интервалы меняются</div><p class="widget-sub">Посмотрите на точки и отрезки. Поднимите уровень с 95% до 99%, сохраняя выборки. Затем увеличьте размер выборки и получите новую серию.</p><div class="d8-controls"><label class="d8-control"><span>Наблюдений n</span><select data-control="n"><option value="100">100</option><option value="5">5</option><option value="20">20</option><option value="400">400</option></select></label><label class="d8-control"><span>Исследований</span><select data-control="M"><option value="500">500</option><option value="100">100</option><option value="2000">2000</option></select></label><label class="d8-control"><span>Уровень доверия</span><select data-control="level"><option value="0.95">95%</option><option value="0.8">80%</option><option value="0.99">99%</option></select></label><label class="d8-control"><span>Источник</span><select data-control="shape"><option value="skew">Асимметричные доставки</option><option value="normal">Нормальная модель для сравнения</option></select></label><button class="btn" data-action="new">Новая серия исследований</button></div><div class="d8-charts"><svg data-chart="forest"></svg></div><div class="readout" data-readout role="status" aria-live="polite"></div><p class="d8-question"><b>Проверь себя.</b> Почему в серии из 100 исследований не обязано быть ровно 95 попаданий? Что даёт более высокий уровень доверия?</p><p class="d8-footnote">Учебная симуляция. Числа округлены; новое повторение может дать другой результат.</p></div>', widget='coverage')))

In [ ]:
#@title Покрытие, ширина и размер выборки { display-mode: "form" }
display(HTML(lesson8_viz('<div class="box def"><div class="t">Покрытие · coverage</div><p>Вероятность того, что построенный по случайной выборке интервал содержит истинный параметр. Долю накрывших параметр интервалов в серии симуляций называем наблюдаемой частотой покрытия.</p></div><div class="box take"><div class="t">Три закономерности</div><ul><li>При фиксированных σ и уровне доверия увеличение n в четыре раза уменьшает ширину нашего интервала вдвое.</li><li>При фиксированных n и σ повышение уровня доверия расширяет интервал.</li><li>Увеличение числа симулированных исследований уточняет оценку покрытия; оно не сужает интервал отдельного исследования.</li></ul></div><div class="box warn"><div class="t">Условия не исчезают</div><p>Заявленные 95% относятся к модели и процедуре. Смещённый отбор, зависимость наблюдений или плохое нормальное приближение могут нарушить покрытие.</p></div><p>Источник: <a href="https://itl.nist.gov/div898/handbook/eda/section3/eda352.htm">NIST — частотный смысл доверительного интервала</a>.</p>')))

**Разобранный пример: 200 исследований в Python.** Цвет каждого интервала определяем сравнением его границ с известной истиной.

In [ ]:
n = 100
experiments = 200
margin = norm.ppf(0.975) * sigma_true / np.sqrt(n)
records = []
for experiment in range(experiments):
    new_sample = 10 + rng.exponential(scale=20, size=n)
    estimate = new_sample.mean()
    low, high = estimate - margin, estimate + margin
    records.append([experiment + 1, estimate, low, high, low <= mu_true <= high])
intervals = pd.DataFrame(records, columns=["experiment", "estimate", "low", "high", "covered"])

fig, ax = plt.subplots(figsize=(10, 6))
for row in intervals.head(40).itertuples():
    color = "#14804a" if row.covered else "#c44555"
    ax.hlines(row.experiment, row.low, row.high, color=color, lw=2)
    ax.scatter(row.estimate, row.experiment, color=color, s=15)
ax.axvline(mu_true, color="#b67a10", ls="--", label="Среднее ГС")
ax.set(xlabel="Среднее время, мин", ylabel="Номер исследования", title="Первые 40 интервалов из 200")
ax.legend()
plt.show()
print(f"Частота покрытия по всем {experiments} исследованиям: {intervals['covered'].mean():.1%}")

## 5. Задача о доставке: что можно сообщить руководителю?

**Самостоятельно.** В следующей ячейке — новая выборка из 100 доставок. В учебной постановке
SD ГС известно и равно 20 минутам, а истинное среднее исследователь не использует.

1. Рассчитайте среднее и приближённый 95%-й доверительный интервал при известном SD ГС.
2. Покажите гистограмму отдельных времён и рядом отдельный график среднего с интервалом.
3. Напишите вывод о среднем времени в двух предложениях. Можно ли этим интервалом обосновать
   обещание «95% заказов доставим внутри указанных границ»? Объясните по графикам.
4. Предскажите, как изменится типичная ширина при 400 наблюдениях. Затем проверьте на новой выборке.

Фраза «среднее истинной ГС попало» здесь недоступна исследователю: для вывода используйте только наблюдения
и известное SD. Генератор нужен нам для учебной проверки, а не как источник готового ответа.

In [ ]:
task_rng = np.random.default_rng(808)
task_delivery = 10 + task_rng.exponential(scale=20, size=100)
task_sigma = 20.0

# TODO: Compute the mean and the standard error under the stated assumptions.
# TODO: Find the interval bounds and draw both requested plots.
# TODO: Explain the result in the next Markdown cell.

**Ваш вывод:** …

In [ ]:
#@title Граница первой встречи { display-mode: "form" }
display(HTML(lesson8_viz('<div class="box take"><div class="t">Что мы уже умеем</div><p>Различаем разброс доставок и точность среднего. Строим интервал при известном SD ГС, объясняем его уровень доверия и проверяем покрытие на симуляциях.</p></div><div class="box idea"><div class="t">Следующий вопрос</div><p>Параметры ГС обычно неизвестны. Как оценить неопределённость, если доступна только одна выборка? И как поступить, если интересует медиана?</p></div><details><summary>Если вместо известной σ подставить выборочное s?</summary>\n<p>Так получают нормальный приближённый интервал с оценённой SE. Для больших подходящих выборок он может\nработать хорошо, но подстановка добавляет неопределённость: точность 95% для малых выборок не гарантирована.\nСпециальные методы на основе t-распределения будем изучать отдельно.</p>\n</details>')))

## 6. Бутстрап: одна выборка, много повторных расчётов

### 6.1. Что можно сделать с уже собранными наблюдениями?

Новые исследования дороги, а сохранённые данные доступны. Попробуем использовать их распределение
как приближение к распределению ГС. Из имеющихся записей будем случайно выбирать столько же записей,
сколько было исходно, **с возвращением**. После каждого выбора запись снова доступна для выбора.

Начнём с 12 наблюдений, чтобы каждое было видно. Это демонстрация механики; маленькая выборка сама по себе
не становится надёжной благодаря бутстрапу.

In [ ]:
#@title Интерактив — Как получается одна бутстрап-выборка { display-mode: "form" }
display(HTML(lesson8_viz('<div class="widget"><div class="widget-title">Как получается одна бутстрап-выборка</div><p class="widget-sub">Сделайте одно повторение и найдите повторившиеся номера. Накопите 200 средних. Затем сравните с отбором всех записей без возвращения.</p><div class="d8-controls"><label class="d8-control"><span>Способ отбора</span><select data-control="replace"><option value="yes">С возвращением — бутстрап</option><option value="no">Без возвращения — сравнение</option></select></label><button class="btn" data-action="one">Одно повторение</button><button class="btn" data-action="many">Ещё 200 повторений</button><button class="btn" data-action="source">Новые 12 исходных наблюдений</button></div><div class="d8-charts"><svg data-chart="original"></svg><svg data-chart="resample"></svg><svg data-chart="bootmeans"></svg></div><div class="readout" data-readout role="status" aria-live="polite"></div><p class="d8-question"><b>Проверь себя.</b> Почему без возвращения среднее не меняется? Сколько новых реальных наблюдений появилось после 200 повторений?</p><p class="d8-footnote">Учебная симуляция. Числа округлены; новое повторение может дать другой результат.</p></div>', widget='resampling')))

In [ ]:
#@title Определение бутстрапа и зачем он нужен { display-mode: "form" }
display(HTML(lesson8_viz('<div class="box def"><div class="t">Бутстрап · bootstrap</div><p>Метод приближённого оценивания выборочного распределения статистики путём многократного создания повторных выборок на основе имеющихся данных и пересчёта статистики.</p></div><div class="box def"><div class="t">Обычный непараметрический бутстрап · nonparametric bootstrap</div><p>Для независимых одинаково распределённых наблюдений получаем повторные выборки того же размера случайным отбором исходных записей с возвращением. Каждая исходная запись имеет одинаковую вероятность выбора. Если значения нескольких записей совпадают, их вероятности складываются.</p></div><div class="box def"><div class="t">Эмпирическое распределение · empirical distribution</div><p>Распределение, которое приписывает наблюдённым значениям вероятности, равные их долям в выборке. Его CDF — знакомая эмпирическая функция распределения. При обычном бутстрапе выбираем из этого распределения.</p></div><div class="box take"><div class="t">Для чего нужен бутстрап</div><p>Для оценки стандартной ошибки, построения доверительных интервалов и изучения того, как меняется результат при повторном сборе данных. Он полезен, когда статистику легко посчитать, а удобная формула её неопределённости неизвестна — например, для медианы.</p></div><div class="box warn"><div class="t">Что мы предполагаем</div><p>Наблюдаемая выборка должна достаточно хорошо описывать интересующее распределение, а схема повторного отбора — сохранять существенное устройство данных. В основной части считаем доставки независимыми. Бутстрап не исправляет смещённый сбор данных.</p></div><p>Объяснение идеи: <a href="https://arxiv.org/html/1411.5279">T. Hesterberg — Bootstrap and Resampling</a>.</p>')))

**Разобранный пример: выбор с возвращением в NumPy.** `rng.choice` выбирает случайные элементы массива.
`size` задаёт число выбранных элементов, `replace=True` разрешает выбирать одну запись повторно.
`np.arange` создаёт номера записей; по ним видно, какие доставки повторились.

In [ ]:
small_sample = delivery_times[:12]
indices = rng.choice(np.arange(len(small_sample)), size=len(small_sample), replace=True)
resampled = small_sample[indices]
display(pd.DataFrame({"Номер исходной записи": indices + 1, "Время, мин": resampled.round(2)}))
print("Среднее исходных 12 записей:", small_sample.mean())
print("Среднее повторной выборки:", resampled.mean())
print("Число разных выбранных записей:", len(np.unique(indices)))

**Самостоятельно:** предскажите результат при `replace=False`, затем проверьте его в предыдущей ячейке. Почему этот опыт не показывает неопределённость среднего?

### 6.2. От повторных средних к стандартной ошибке

Фиксируем исходные 100 наблюдений. Для каждой повторной выборки сохраняем **одно число** — её среднее.
Разброс этих чисел используем как оценку стандартной ошибки.

Обратите внимание: бутстрап-средние располагаются около **исходного выборочного среднего**.
Оно не обязано совпадать с истинным средним ГС.

In [ ]:
bootstrap_rng = np.random.default_rng(8008)
B = 2000
boot_means = []
for _ in range(B):
    repeated_sample = bootstrap_rng.choice(delivery_times, size=len(delivery_times), replace=True)
    boot_means.append(repeated_sample.mean())
boot_means = np.array(boot_means)

se_boot = boot_means.std(ddof=1)
se_formula = delivery_times.std(ddof=1) / np.sqrt(len(delivery_times))
ax = sns.histplot(boot_means, bins=35, color="#14804a")
ax.axvline(delivery_times.mean(), color="#14804a", lw=2, label="Исходное выборочное среднее")
ax.axvline(mu_true, color="#b67a10", ls="--", label="Истинное среднее — только для проверки")
ax.set(xlabel="Среднее бутстрап-выборки, мин", ylabel="Число повторений")
ax.legend()
plt.show()
print(f"SE по бутстрапу: {se_boot:.3f} мин; оценка s / √n: {se_formula:.3f} мин")

In [ ]:
#@title Что именно мы оценили { display-mode: "form" }
display(HTML(lesson8_viz('<div class="box take"><div class="t">SE — разброс оценок</div><p>Стандартное отклонение исходных времён описывает различия между доставками. Стандартное отклонение бутстрап-средних оценивает неопределённость среднего. Не смешивайте два массива.</p></div><div class=\'d8-equation\'>$$\\widehat{SE}_{\\mathrm{boot}}=\\operatorname{SD}(\\bar x_1^*,\\ldots,\\bar x_B^*).$$</div><div class=\'where\'><b>Обозначения.</b> $\\widehat{SE}_{\\mathrm{boot}}$ — бутстрап-оценка стандартной ошибки; $\\bar x_b^*$ — среднее b-й повторной выборки; звёздочка отмечает бутстрап; $B$ — число повторений; $\\operatorname{SD}$ — выборочное стандартное отклонение с делителем B − 1.</div><div class="box idea"><div class="t">n и B отвечают за разное</div><p><b>n</b> — сколько исходных наблюдений мы собрали. <b>B</b> — сколько повторных выборок посчитал компьютер. Увеличение B уменьшает вычислительные колебания оценки SE и квантилей, но не добавляет информации о ГС.</p></div><details><summary>Почему две оценки SE близки, но не обязаны совпасть?</summary>\n<p>Бутстрап использует конечное эмпирическое распределение, а формула с выборочным SD содержит поправку\nчерез делитель n − 1. Дополнительно влияет конечное число бутстрап-повторений. Сравниваем близость,\nа не требуем равенства до последнего знака.</p></details>')))

### 6.3. Простой интервал по квантилям

Возьмём квантили 2,5% и 97,5% **бутстрап-значений статистики**. Это границы центральных 95%.
Такой способ называют **квантильным, или процентильным, бутстрап-интервалом (percentile interval)**.
Он прост для знакомства; его фактическую надёжность мы проверим в следующем опыте.

In [ ]:
#@title Формула квантильного интервала { display-mode: "form" }
display(HTML(lesson8_viz('<div class=\'d8-equation\'>$$I_{\\mathrm{perc}}=[q^*_{0.025},\\ q^*_{0.975}].$$</div><div class=\'where\'><b>Обозначения.</b> $I_{\\mathrm{perc}}$ — приближённый 95%-й процентильный интервал; $q^*_{0.025}$ и $q^*_{0.975}$ — квантили уровней 2,5% и 97,5% распределения бутстрап-оценок. Звёздочка обозначает бутстрап; это квантили средних (или другой выбранной статистики), а не исходных наблюдений.</div><div class="box warn"><div class="t">Две разные доли</div><p>Мы берём центральные 95% <b>бутстрап-оценок</b>. Это ещё не доказывает, что полученный метод накрывает <b>истинный параметр</b> в 95% новых исследований. Для такого вывода нужны условия и проверка точности приближения.</p></div>')))

In [ ]:
#@title Интерактив — Среднее и доверительный интервал по одной выборке { display-mode: "form" }
display(HTML(lesson8_viz('<div class="widget"><div class="widget-title">Среднее и доверительный интервал по одной выборке</div><p class="widget-sub">Сначала меняйте только B и повторяйте бутстрап. Затем получите новую исходную выборку. Откройте истину для проверки. Медиану можно исследовать дополнительно.</p><div class="d8-controls"><label class="d8-control"><span>Исходных доставок n</span><select data-control="n"><option value="100">100</option><option value="20">20</option><option value="50">50</option><option value="400">400</option></select></label><label class="d8-control"><span>Повторений B</span><select data-control="B"><option value="2000">2000</option><option value="200">200</option><option value="5000">5000</option></select></label><label class="d8-control"><span>Уровень интервала</span><select data-control="level"><option value="0.95">95%</option><option value="0.8">80%</option><option value="0.99">99%</option></select></label><label class="d8-control"><span>Что оцениваем</span><select data-control="stat"><option value="mean">Среднее</option><option value="median">Медиана — дополнительный опыт</option></select></label><label class="d8-control"><span>Истинный ответ</span><select data-control="truth"><option value="hide">Скрыт</option><option value="show">Показать для проверки</option></select></label><button class="btn" data-action="repeat">Повторить бутстрап</button><button class="btn" data-action="source">Новая исходная выборка</button></div><div class="d8-charts"><svg data-chart="data"></svg><svg data-chart="distribution"></svg><svg data-chart="interval"></svg></div><div class="readout" data-readout role="status" aria-live="polite"></div><p class="d8-question"><b>Проверь себя.</b> Какая кнопка создаёт новые наблюдения? Почему 5000 повторений не равны 5000 изученным доставкам?</p><p class="d8-footnote">Учебная симуляция. Числа округлены; новое повторение может дать другой результат.</p></div>', widget='bootstrap')))

**Разобранный пример: границы интервала и два обычных графика.** Данные для гистограммы — сохранённые `boot_means`, а не новые доставки.

In [ ]:
boot_low, boot_high = np.quantile(boot_means, [0.025, 0.975])
estimate = delivery_times.mean()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(boot_means, bins=35, color="#14804a", ax=axes[0])
axes[0].axvspan(boot_low, boot_high, color="#14804a", alpha=0.15)
axes[0].axvline(estimate, color="#14804a", lw=2)
axes[0].set(xlabel="Среднее бутстрап-выборки, мин", ylabel="Число повторений",
            title="Центральные 95% бутстрап-средних")
axes[1].hlines(0, boot_low, boot_high, color="#14804a", lw=5)
axes[1].scatter([estimate], [0], color="#14804a", s=80, label="Среднее выборки")
axes[1].axvline(mu_true, color="#b67a10", ls="--", label="Истина — для проверки")
axes[1].set(xlabel="Среднее время, мин", yticks=[], ylim=(-1, 1),
            title=f"Интервал [{boot_low:.2f}; {boot_high:.2f}]")
axes[1].legend()
plt.tight_layout()
plt.show()

### 6.4. Проверяем покрытие бутстрап-интервала

Здесь два уровня повторения. **Внешний:** собираем совершенно новую исходную выборку.
**Внутренний:** для этой одной выборки делаем бутстрап и получаем один интервал.

Сначала предскажите: будет ли метод, который берёт центральные 95% бутстрап-средних,
всегда давать около 95% покрытия истинного среднего? Проверьте на малых и более крупных выборках.

In [ ]:
#@title Интерактив — Много новых исследований, внутри каждого — свой бутстрап { display-mode: "form" }
display(HTML(lesson8_viz('<div class="widget"><div class="widget-title">Много новых исследований, внутри каждого — свой бутстрап</div><p class="widget-sub">Начните с n = 10. Увеличьте размер до 100 и повторите серию. Сравните частоты покрытия, не только отдельные отрезки.</p><div class="d8-controls"><label class="d8-control"><span>Доставок в каждом исследовании</span><select data-control="n"><option value="10">10</option><option value="30">30</option><option value="100">100</option></select></label><label class="d8-control"><span>Новых исследований</span><select data-control="M"><option value="100">100</option><option value="300">300</option></select></label><button class="btn" data-action="new">Повторить серию исследований</button></div><div class="d8-charts"><svg data-chart="known"></svg><svg data-chart="bootstrap"></svg></div><div class="readout" data-readout role="status" aria-live="polite"></div><p class="d8-question"><b>Проверь себя.</b> Почему нельзя оценить покрытие, просто многократно выполняя бутстрап одной и той же исходной выборки?</p><p class="d8-footnote">Учебная симуляция. Числа округлены; новое повторение может дать другой результат.</p></div>', widget='bootcoverage')))

In [ ]:
#@title Что считать проверкой метода { display-mode: "form" }
display(HTML(lesson8_viz('<div class="box take"><div class="t">В симуляции мы знаем истину</div><p>Для оценки покрытия повторяем весь путь: новая исходная выборка → её бутстрап → один интервал → проверка попадания истины. Один удачный интервал не доказывает надёжность метода.</p></div><div class="box warn"><div class="t">Ограничения обычного бутстрапа</div><ul><li>Малая выборка может плохо представлять ГС, особенно редкие большие значения.</li><li>Повторный отбор не исправляет исходное смещение и не восстанавливает события, которых мы не наблюдали.</li><li>Зависимость требует подходящей схемы отбора.</li><li>Для крайних квантилей, максимума и других чувствительных статистик простой бутстрап может работать плохо. Его универсальность не означает автоматическую точность.</li></ul></div><p>Источник: <a href="https://research.google/pubs/what-teachers-should-know-about-the-bootstrap-resampling-in-the-undergraduate-statistics-curriculum/">Hesterberg — точность бутстрап-интервалов, особенно на малых выборках</a>.</p>')))

**Разобранный пример: два цикла.** Внешний цикл создаёт 120 новых исследований.
Внутренний делает 400 повторных выборок для каждого из них. Это небольшой учебный расчёт:
его частота покрытия и квантили ещё заметно колеблются.

In [ ]:
coverage_rng = np.random.default_rng(8064)
outer_experiments = 120
inner_repeats = 400
coverage_n = 30
coverage_records = []
for experiment in range(outer_experiments):
    observed = 10 + coverage_rng.exponential(scale=20, size=coverage_n)
    repeated_means = []
    for _ in range(inner_repeats):
        repeated = coverage_rng.choice(observed, size=coverage_n, replace=True)
        repeated_means.append(repeated.mean())
    low, high = np.quantile(repeated_means, [0.025, 0.975])
    coverage_records.append([experiment + 1, observed.mean(), low, high, low <= mu_true <= high])
boot_intervals = pd.DataFrame(coverage_records, columns=["experiment", "estimate", "low", "high", "covered"])

fig, ax = plt.subplots(figsize=(10, 6))
for row in boot_intervals.head(40).itertuples():
    color = "#14804a" if row.covered else "#c44555"
    ax.hlines(row.experiment, row.low, row.high, color=color)
    ax.scatter(row.estimate, row.experiment, color=color, s=15)
ax.axvline(mu_true, color="#b67a10", ls="--")
ax.set(xlabel="Среднее время, мин", ylabel="Исследование", title="Бутстрап-интервалы: первые 40 исследований")
plt.show()
print(f"Частота покрытия по {outer_experiments} исследованиям: {boot_intervals['covered'].mean():.1%}")

### 6.5. После ручного цикла — библиотечная функция

В `scipy.stats.bootstrap` передаём данные, функцию статистики и число повторений.
`(delivery_times,)` — кортеж с одной выборкой. Явно выбираем `method="percentile"`, чтобы повторить
изученный алгоритм: значение по умолчанию у SciPy — другой метод, BCa, с которым познакомимся обзорно ниже.
Числа могут немного отличаться от нашего цикла из-за других случайных повторных выборок.

In [ ]:
result = bootstrap(
    (delivery_times,), np.mean,
    n_resamples=2000, confidence_level=0.95, method="percentile",
    random_state=np.random.default_rng(865),
)
print("Стандартная ошибка:", result.standard_error)
print("Границы интервала:", result.confidence_interval.low, result.confidence_interval.high)

## 7. Какие бывают бутстрапы и зачем нужны варианты?

Здесь важно различить два выбора: **как получать повторные выборки** и **как по ним строить интервал**.
Ниже — карта методов для ориентировки. Для основной практики достаточно обычного бутстрапа
независимых наблюдений и интервала по квантилям. Дополнительные интерактивы можно оставить на семинар.

In [ ]:
#@title Откуда берутся повторные данные { display-mode: "form" }
display(HTML(lesson8_viz('\n<div class="d8-table"><table>\n<tr><th>Вариант</th><th>Что делаем</th><th>Когда и зачем</th></tr>\n<tr><td><b>Непараметрический</b><br>nonparametric</td><td>Выбираем наблюдённые записи с возвращением.</td><td>Оцениваем неопределённость без задания конкретного семейства распределений. Для обычного отбора строк предполагаем их независимость.</td></tr>\n<tr><td><b>Параметрический</b><br>parametric</td><td>Выбираем семейство распределений, оцениваем его параметры по данным и генерируем новые выборки из подобранной модели.</td><td>Когда модель распределения обоснована. Результат зависит от правильности этой модели.</td></tr>\n</table></div>\n<p>Например, среднее и SD можно оценить по выборке и генерировать нормальные данные с этими параметрами.\nТак мы добавляем предположение о форме распределения. Большое число повторений не проверяет это предположение.</p>\n<div class="box warn"><div class="t">Непараметрический не означает «без условий»</div><p>Мы не задаём конкретную форму распределения ГС, но по-прежнему учитываем отбор данных, зависимость, размер выборки и свойства вычисляемой статистики.</p></div>')))

In [ ]:
#@title Интерактив — Выбирать наблюдения или генерировать из модели? { display-mode: "form" }
display(HTML(lesson8_viz('<details><summary>Дополнительно: Выбирать наблюдения или генерировать из модели?</summary><div class="widget"><div class="widget-title">Выбирать наблюдения или генерировать из модели?</div><p class="widget-sub">Переключите способ генерации. Сравните отдельные значения на верхних графиках и средние внизу. Найдите проблему нормальной модели для этих доставок.</p><div class="d8-controls"><label class="d8-control"><span>Источник повторных выборок</span><select data-control="kind"><option value="empirical">Наблюдаемые записи</option><option value="normal">Подобранная нормальная модель</option></select></label><button class="btn" data-action="new">Новые исходные данные</button></div><div class="d8-charts"><svg data-chart="observed"></svg><svg data-chart="generated"></svg><svg data-chart="statistics"></svg></div><div class="readout" data-readout role="status" aria-live="polite"></div><p class="d8-question"><b>Проверь себя.</b> Почему гладкий вид гистограммы сам по себе не подтверждает правильность модели?</p><p class="d8-footnote">Учебная симуляция. Числа округлены; новое повторение может дать другой результат.</p></div></details>', widget='model')))

In [ ]:
#@title Что пересэмплировать целиком { display-mode: "form" }
display(HTML(lesson8_viz('\n<p><b>Пересэмплирование (resampling)</b> — повторный случайный отбор данных по заданной схеме.\nЕдиницу отбора определяет устройство исследования.</p>\n<div class="d8-table"><table>\n<tr><th>Схема</th><th>Единица повторного отбора</th><th>Пример и причина</th></tr>\n<tr><td><b>Парный</b><br>paired</td><td>Пара измерений одного объекта.</td><td>«До» и «после» для одного пациента выбираем вместе; независимо перемешивать два столбца нельзя.</td></tr>\n<tr><td><b>Кластерный</b><br>cluster</td><td>Целая группа связанных наблюдений. Такая группа называется кластером.</td><td>Выбираем дни вместе со всеми их поездками, если дни можно считать независимыми. Сохраняем связь внутри дня.</td></tr>\n<tr><td><b>Блочный</b><br>block</td><td>Последовательный отрезок временного ряда — блок.</td><td>При связи соседних измерений сохраняем их локальный порядок. Длина блока влияет на результат.</td></tr>\n</table></div>\n<p>Эти схемы описывают, какие связи сохранить. Например, кластерный бутстрап можно делать\nнепараметрически: выбирать с возвращением наблюдённые кластеры.</p>\n<details><summary>Дополнительно: стратифицированный бутстрап</summary>\n<p><b>Страта (stratum)</b> — заранее выделенная подгруппа, например район города в плане обследования.\nПри стратифицированном бутстрапе повторный отбор проводят отдельно внутри страт, сохраняя предусмотренную\nпланом структуру выборки. При расчёте итогов могут понадобиться веса. Это другая причина группировки,\nчем зависимость наблюдений внутри кластера.</p></details>\n<p>Документация: <a href="https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html">SciPy — парный отбор</a>;\n<a href="https://stat.ethz.ch/R-manual/R-devel/library/boot/html/boot.html">R boot — схемы генерации и страты</a>;\n<a href="https://bashtage.github.io/arch/bootstrap/timeseries-bootstraps.html">arch — блочный бутстрап временных рядов</a>.</p>\n')))

In [ ]:
#@title Интерактив — Почему поездки одного дня нельзя всегда перемешивать по одной { display-mode: "form" }
display(HTML(lesson8_viz('<details><summary>Дополнительно: Почему поездки одного дня нельзя всегда перемешивать по одной</summary><div class="widget"><div class="widget-title">Почему поездки одного дня нельзя всегда перемешивать по одной</div><p class="widget-sub">Посмотрите на группы на верхнем графике. Увеличьте общий сдвиг внутри дня. Сравните ширину двух распределений средних; их горизонтальные шкалы одинаковы.</p><div class="d8-controls"><label class="d8-control"><span>Независимых дней</span><select data-control="groups"><option value="12">12</option><option value="6">6</option><option value="30">30</option></select></label><label class="d8-control"><span>SD общего сдвига дня</span><input type="range" data-control="tau" min="0" max="10" step="0.5" value="5"><output data-value="tau">5</output></label><button class="btn" data-action="new">Новые данные</button></div><div class="d8-charts"><svg data-chart="groups"></svg><svg data-chart="rows"></svg><svg data-chart="whole"></svg></div><div class="readout" data-readout role="status" aria-live="polite"></div><p class="d8-question"><b>Проверь себя.</b> Что теряется при отборе отдельных строк? Почему тысячи повторений не заменяют большое число независимых дней?</p><p class="d8-footnote">Учебная симуляция. Числа округлены; новое повторение может дать другой результат.</p></div></details>', widget='clusters')))

In [ ]:
#@title Как получить границы из бутстрап-распределения { display-mode: "form" }
display(HTML(lesson8_viz('\n<div class="d8-table"><table>\n<tr><th>Метод интервала</th><th>Идея</th></tr>\n<tr><td><b>Percentile</b> — квантильный</td><td>Берём соответствующие квантили бутстрап-оценок. Этот способ уже реализовали вручную.</td></tr>\n<tr><td><b>Basic</b> — базовый, обратный процентильный</td><td>Отражаем границы квантильного интервала относительно исходной оценки. Так переносим оценённые ошибки на неизвестный параметр.</td></tr>\n<tr><td><b>BCa</b> — bias-corrected and accelerated</td><td>Используем скорректированные уровни квантилей с учётом оценки смещения и изменения разброса статистики. Это более сложная процедура; здесь достаточно знать её назначение.</td></tr>\n</table></div>\n<p><b>Смещение оценки (bias)</b> — разность между средним значением оценки по повторным исследованиям\nи истинным параметром. Это свойство процедуры, а не ошибка одного наблюдения.</p>\n<div class="box take"><div class="t">Схема выборок и метод интервала — разные настройки</div><p>Например, обычный непараметрический бутстрап может использовать интервалы percentile, basic или BCa. BCa не означает иной способ выбирать строки. Ни один вариант автоматически не исправляет неподходящие данные или нарушенные условия.</p></div><details><summary>Дополнительно: формула basic-интервала</summary><div class=\'d8-equation\'>$$I_{\\mathrm{basic}}=[2\\widehat\\theta-q^*_{0.975},\\ 2\\widehat\\theta-q^*_{0.025}].$$</div><div class=\'where\'><b>Обозначения.</b> $I_{\\mathrm{basic}}$ — базовый приближённый 95%-й интервал; $\\widehat\\theta$ — исходная оценка параметра (например, среднее выборки); $q^*_{0.025}$ и $q^*_{0.975}$ — квантили бутстрап-оценок; звёздочка обозначает бутстрап. Меньшая граница получается из верхней квантили.</div></details><p>Источник: <a href="https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html">SciPy — методы percentile, basic и BCa</a>.</p>')))

In [ ]:
#@title Дополнительно — сравниваем методы интервала на одних данных { display-mode: "form" }
method_rows = []
for method in ["percentile", "basic", "BCa"]:
    result = bootstrap((delivery_times,), np.mean, n_resamples=2000,
                       method=method, confidence_level=0.95,
                       random_state=np.random.default_rng(877))
    method_rows.append([method, result.confidence_interval.low, result.confidence_interval.high])
method_intervals = pd.DataFrame(method_rows, columns=["method", "low", "high"])
fig, ax = plt.subplots(figsize=(10, 3))
for index, row in enumerate(method_intervals.itertuples()):
    ax.hlines(index, row.low, row.high, color="#14804a", lw=4)
    ax.scatter(delivery_times.mean(), index, color="#14804a", s=45)
ax.axvline(mu_true, color="#b67a10", ls="--", label="Истина — для проверки")
ax.set(yticks=range(3), yticklabels=method_intervals["method"], xlabel="Среднее время, мин",
       title="Одни данные и повторные выборки, разные правила границ")
ax.legend()
plt.show()
display(method_intervals)

**Обсудите:** можно ли выбрать лучший метод только по тому, какой из этих трёх интервалов оказался короче? Какая симуляция нужна для проверки надёжности?

## 8. Практика и проверка понимания

**Задача A. Что изменилось?** Зафиксируйте выборку в интерактиве 6.3. Увеличьте B с 200 до 5000,
затем отдельно увеличьте n. Объясните различие действий и приложите два наблюдения с графиков.

**Задача B. Медиана доставок.** Для `task_delivery` из раздела 5 постройте бутстрап-распределение медианы
и процентильный 95%-й интервал. Сначала реализуйте цикл самостоятельно, затем проверьте через библиотеку.
Формулу SE среднего к медиане не применяйте. Интерпретируйте, какой параметр оцениваете.

**Задача C. Проверка обещания 95%.** В коде 6.4 сравните размеры исходной выборки 10 и 100.
Запишите частоты покрытия. Объясните, почему из одной конечной серии нельзя требовать точного совпадения с 95%.

**Задача D. Перенос на автобусы — словами.** В поездках одного дня есть общий фактор пробок.
Предложите единицу повторного отбора и назовите условие, при котором такая схема разумна.
Отдельно рассмотрите случай, когда соседние дни тоже связаны. Данных для расчёта здесь нет: используйте
синтетический пример из раздела 7 и знания о сборе автобусного практикума.

In [ ]:
# TODO: Bootstrap the median of task_delivery and draw its distribution and interval.
# TODO: Compare the custom loop with scipy.stats.bootstrap using the same statistic.

**Ваши выводы по A–D:** …

In [ ]:
#@title Короткие формулировки для устного ответа { display-mode: "form" }
display(HTML(lesson8_viz('\n<div class="d8-table"><table>\n<tr><th>Понятие</th><th>Что нужно уметь сказать</th></tr>\n<tr><td>Доверительный интервал</td><td>Интервал по выборке, построенный процедурой с заданной вероятностью покрытия неизвестного параметра.</td></tr>\n<tr><td>Уровень доверия 95%</td><td>При повторении исследования процедура накрывает истинный параметр примерно в 95% случаев, если выполнены её условия.</td></tr>\n<tr><td>Бутстрап</td><td>Приближение выборочного распределения статистики повторными выборками на основе наблюдаемых данных.</td></tr>\n<tr><td>Обычный непараметрический бутстрап</td><td>Отбор исходных наблюдений с возвращением, с сохранением размера выборки, и пересчёт статистики.</td></tr>\n<tr><td>Бутстрап-SE</td><td>Стандартное отклонение полученных бутстрап-оценок.</td></tr>\n<tr><td>Percentile-интервал</td><td>Интервал между соответствующими квантилями бутстрап-оценок; его покрытие приближённое.</td></tr>\n</table></div>\n<div class="box take"><div class="t">Один вывод о точности</div><p>Сообщайте оценку, интервал, способ его построения и основные условия. По одному среднему или одной ширине интервала надёжность исследования не определяется.</p></div>')))

## 9. Подводка: что делать с утверждением «среднее равно 30»?

Служба доставки утверждает: истинное среднее равно 30 минутам.
**Статистическая гипотеза** — предположение о распределении или его параметрах, которое проверяют по данным.
Здесь гипотеза относится к среднему ГС.

На графике уже можно отметить 30 и увидеть его положение относительно интервала.
Попадание 30 внутрь интервала не доказывает точного равенства. Промах требует оценивать результат с учётом
модели и надёжности процедуры: случайные промахи встречаются даже при верных условиях.

Дальше нам нужен способ ответить: **насколько необычен наблюдаемый результат, если проверяемое предположение верно?**
На следующем занятии введём p-value, правила проверки гипотез и статистическую значимость.
В этом колабе такие решения не принимаем.

In [ ]:
#@title Бутстрап и проверяемое предположение { display-mode: "form" }
display(HTML(lesson8_viz('<div class="box idea"><div class="t">Что важно сохранить к следующему занятию</div><p>Обычный бутстрап среднего даёт распределение около наблюдаемого среднего. Если оно равно 33, центр бутстрап-распределения тоже будет около 33. Это само по себе не моделирует предположение «истинное среднее равно 30». Для проверки гипотез потребуется отдельно учесть проверяемое предположение.</p></div>')))

**Источники и документация**

- [NIST: смысл доверительного интервала и формула при известном SD ГС](https://www.itl.nist.gov/div898/handbook/prc/section1/prc14.htm).
- [NIST: интерпретация покрытия при повторном сборе данных](https://itl.nist.gov/div898/handbook/eda/section3/eda352.htm).
- [Hesterberg: как преподавать бутстрап, возможности и ограничения](https://arxiv.org/html/1411.5279).
- [SciPy: bootstrap, парные выборки, SE и методы интервалов](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html).
- [NumPy: случайный отбор с возвращением, Generator.choice](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.choice.html).
- [Seaborn: гистограммы, histplot](https://seaborn.pydata.org/generated/seaborn.histplot.html).
- [R boot: параметрическая генерация, страты и другие схемы](https://stat.ethz.ch/R-manual/R-devel/library/boot/html/boot.html).
- [arch: блочный бутстрап временных рядов](https://bashtage.github.io/arch/bootstrap/timeseries-bootstraps.html).

Все вычисления работают на синтетике без загрузки датасетов. Числа в интерактивах и Python могут различаться:
они используют разные случайные выборки. Для старых версий SciPy в примерах использован совместимый
аргумент `random_state`; в новых версиях его заменяет `rng`.
В локальном Jupyter может потребоваться доверить ноутбук через меню Trust Notebook.